In [ ]:
# =============================================================================
# MODEL VALIDATION FRAMEWORK FOR iPS CELL CLASSIFICATION BY JOVIAL
# =============================================================================
# This notebook performs three validation experiments 
#
# 1. CROSS-VALIDATION: 5-fold × 5 repeats on the optimal window for each phase
#    - Phase 1: Start frame 12, length 29 (frames 12-40)
#    - Phase 2: Start frame 108, length 29 (frames 108-136)
#    - Phase 3: Start frame 152, length 29 (frames 152-180)
#    - Phase 4: Start frame 233, length 29 (frames 233-261)
#
# 2. HOLDOUT VALIDATION: Train on optimal window, test on neighboring windows
#    - Phase 1: Train on I=12, test on I-3=9, I-2=10, I-1=11, I=12, I+1=13, I+2=14, I+3=15
#    - Phase 2: Train on I=108, test on I-3=105, I-2=106, I-1=107, I=108, I+1=109, I+2=110, I+3=111
#    - Phase 3: Train on I=152, test on I-3=149, I-2=150, I-1=151, I=152, I+1=153, I+2=154, I+3=155
#    - Phase 4: Train on I=233, test on I-3=230, I-2=231, I-1=232, I=233, I+1=234, I+2=235, I+3=236
#
# 3. INDEPENDENT TEST: Train on one phase, test on distant windows from next phase (or previous phase for Phase 4)
#    - Train Phase 1 (I=12), test on Phase 2 windows: 102, 105, 108, 111, 114, 117, 120
#    - Train Phase 2 (I=108), test on Phase 3 windows: 143, 146, 149, 152, 155, 158, 161
#    - Train Phase 3 (I=152), test on Phase 4 windows: 233, 236, 239, 242, 245, 248, 251
#    - Train Phase 4 (I=233), test on Phase 1 windows: 3, 6, 9, 12, 15, 18, 21
# =============================================================================

In [ ]:
# =============================================================================
# CELL 1: IMPORTS AND DIRECTORY SETUP
# =============================================================================

import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import json
import time
from datetime import datetime
import pickle
from collections import defaultdict, Counter
import itertools
from scipy import stats

warnings.filterwarnings('ignore')

# ---------- Scikit-learn ----------
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, matthews_corrcoef,
    balanced_accuracy_score, average_precision_score, fbeta_score
)
from sklearn.ensemble import (
    GradientBoostingClassifier, VotingClassifier,
    RandomForestClassifier, ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

# ---------- Optional boosting libs ----------
try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
    print("LightGBM available")
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM NOT available - install with: pip install lightgbm")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("XGBoost available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost NOT available")

# ---------- Visualisation ----------
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.ticker import MultipleLocator
import seaborn as sns

# ---------- Paths ----------
BASE_PATH       = "."
CONSENSUS_PATH  = os.path.join(BASE_PATH, "consensus")
RESULTS_DIR     = "results"
MODELS_DIR      = os.path.join(RESULTS_DIR, "models")
PLOTS_DIR       = os.path.join(RESULTS_DIR, "plots")
DATA_DIR        = os.path.join(RESULTS_DIR, "data")
LOGS_DIR        = os.path.join(RESULTS_DIR, "logs")

for _d in [RESULTS_DIR, MODELS_DIR, PLOTS_DIR, DATA_DIR, LOGS_DIR]:
    os.makedirs(_d, exist_ok=True)

print(f"\nResults directory : {RESULTS_DIR}")
print(f"Plots directory   : {PLOTS_DIR}")
print(f"Data directory    : {DATA_DIR}")



In [ ]:
# =============================================================================
# CELL 2: GLOBAL CONFIGURATION
# =============================================================================

# ---------- Phase definitions (identical to reference) ----------
PHASES = {
    1: {'frames': (0, 101),   'start_time': '0h',      'end_time': '16h50min',
        'interval': '10min',  'duration': '16h50min',   'total_frames': 102},
    2: {'frames': (102, 142), 'start_time': '18h',     'end_time': '24h40min',
        'interval': '10min',  'duration': '6h40min',    'total_frames': 41},
    3: {'frames': (143, 232), 'start_time': '25h50min','end_time': '40h40min',
        'interval': '10min',  'duration': '14h50min',   'total_frames': 90},
    4: {'frames': (233, 274), 'start_time': '41h50min','end_time': '48h40min',
        'interval': '10min',  'duration': '6h50min',    'total_frames': 42},
}

# ---------- Best window configs from time-window selection ----------
BEST_CONFIGS = {
    1: {'start_frame': 12,  'window_length': 29, 'time': '2h/12',
        'avg_mcc': 0.8426},
    2: {'start_frame': 108, 'window_length': 29, 'time': '19h/108',
        'avg_mcc': 0.7727},
    3: {'start_frame': 152, 'window_length': 29, 'time': '27h20min/152',
        'avg_mcc': 0.7217},
    4: {'start_frame': 233, 'window_length': 29, 'time': '41h50min/233',
        'avg_mcc': 0.6302},
}

WINDOW_LENGTH = 29   # optimal window length from time-window selection

# ---------- Holdout window offsets (I-3 … I+3) ----------
HOLDOUT_CONFIGS = {
    1: {'train_start': 12,
        'test_starts': [9, 10, 11, 12, 13, 14, 15],
        'offsets':     [-3, -2, -1, 0, 1, 2, 3],
        'train_window':'12-40',
        'test_windows':['9-37','10-38','11-39','12-40','13-41','14-42','15-43']},
    2: {'train_start': 108,
        'test_starts': [105,106,107,108,109,110,111],
        'offsets':     [-3,-2,-1,0,1,2,3],
        'train_window':'108-136',
        'test_windows':['105-133','106-134','107-135','108-136',
                        '109-137','110-138','111-139']},
    3: {'train_start': 152,
        'test_starts': [149,150,151,152,153,154,155],
        'offsets':     [-3,-2,-1,0,1,2,3],
        'train_window':'146-174',
        'test_windows':['149-177','150-178','151-179','152-180',
                        '153-181','154-182','155-183']},
    4: {'train_start': 233,
        'test_starts': [230,231,232,233,234,235,236],
        'offsets':     [-3,-2,-1,0,1,2,3],
        'train_window':'233-261',
        'test_windows':['230-258','231-259','232-260','233-261',
                        '234-262','235-263','236-264']},
}

# ---------- Independent test configs (exact plan: step-by-3, distant windows) ----------
# Train Phase P optimal window → test on DISTANT windows from next phase (or previous phase for Phase 4)
# Step = 3 frames apart
INDEPENDENT_CONFIGS = {
    1: {'train_phase':1,  'train_start':12,   'train_window':'12-40',
        'test_phase':2,
        'test_starts': [102, 105, 108, 111, 114, 117, 120],
        'test_windows':['102-130','105-133','108-136','111-139','114-142','117-145','120-148'],
        'test_labels': ['S11','S12','S13','S14','S15','S16','S17']},
    2: {'train_phase':2,  'train_start':108,  'train_window':'108-136',
        'test_phase':3,
        'test_starts': [143, 146, 149, 152, 155, 158, 161],
        'test_windows':['143-171','146-174','149-177','152-180','155-183','158-186','161-189'],
        'test_labels': ['S21','S22','S23','S24','S25','S26','S27']},
    3: {'train_phase':3,  'train_start':152,  'train_window':'152-180',
        'test_phase':4,
        'test_starts': [233, 236, 239, 242, 245, 248, 251],
        'test_windows':['233-261','236-264','239-267','242-270','245-273','248-276','251-279'],
        'test_labels': ['S31','S32','S33','S34','S35','S36','S37']},
    4: {'train_phase':4,  'train_start':233,  'train_window':'233-261',
        'test_phase':1,
        'test_starts': [3, 6, 9, 12, 15, 18, 21],
        'test_windows':['3-31','6-34','9-37','12-40','15-43','18-46','21-49'],
        'test_labels': ['S41','S42','S43','S44','S45','S46','S47']},
}

# ---------- Feature categories----------
FEATURE_CATEGORIES = {
    'morphological': ['Area', 'Volume', 'Perimeter'],
    'shape':         ['Compactness', 'Sphericity', 'Solidity', 'Extent',
                      'Ellipsoid-Prolate', 'Ellipsoid-Oblate'],
    'biological':    ['Nucleus-Cytoplasm Volume Ratio'],
    'motion':        ['Speed', 'Displacement'],
    'intensity':     ['Intensity-Mean', 'Intensity-Sum', 'Intensity-StdDev',
                      'Intensity-Max', 'Intensity-Min'],
}
ALL_FEATURES = [f for cat in FEATURE_CATEGORIES.values() for f in cat]

# =============================================================================
# MODEL CONFIGURATIONS FOR ALL MODELS
# =============================================================================

# XGBOOST CONFIGURATION
XGB_CONFIG = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'max_depth': 5,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'scale_pos_weight': 5,
    'eval_metric': 'aucpr',
    'random_state': 42,
    'verbosity': 0,
    'use_label_encoder': False
}

# LIGHTGBM CONFIGURATION
LGBM_CONFIG = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'max_depth': 5,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 3,
    'min_data_in_leaf': 10,
    'scale_pos_weight': 5,
    'objective': 'binary',
    'metric': 'auc',
    'random_state': 42,
    'verbosity': -1
}

# RANDOM FOREST CONFIGURATION
RF_CONFIG = {
    'n_estimators': 300,
    'max_depth': 15,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'class_weight': 'balanced_subsample',
    'bootstrap': True,
    'random_state': 42,
    'n_jobs': -1
}

# GRADIENT BOOSTING CONFIGURATION
GB_CONFIG = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'max_depth': 5,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'subsample': 0.8,
    'random_state': 42
}

# CATBOOST CONFIGURATION
CATBOOST_CONFIG = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'depth': 5,
    'l2_leaf_reg': 1.0,
    'bootstrap_type': 'Bayesian',
    'bagging_temperature': 1.0,
    'od_type': 'Iter',
    'od_wait': 20,
    'random_seed': 42,
    'logging_level': 'Silent',
    'allow_writing_files': False,
    'auto_class_weights': 'Balanced'
}

# MLP CONFIGURATION
MLP_CONFIG = {
    'hidden_layer_sizes': (100, 50),
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.001,
    'learning_rate': 'adaptive',
    'learning_rate_init': 0.01,
    'max_iter': 500,
    'early_stopping': True,
    'validation_fraction': 0.2,
    'n_iter_no_change': 15,
    'random_state': 42
}

# ADABOOST CONFIGURATION
ADA_CONFIG = {
    'n_estimators': 300,
    'learning_rate': 0.1,
    'algorithm': 'SAMME.R',
    'random_state': 42
}


# KNN CONFIGURATION
KNN_CONFIG = {
    'n_neighbors': 5,
    'weights': 'distance',
    'n_jobs': -1
}

# LOGISTIC REGRESSION CONFIGURATION
LR_CONFIG = {
    'C': 1.0,
    'max_iter': 1000,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1
}

# BALANCED BAGGING CONFIGURATION
BALANCED_BAGGING_CONFIG = {
    'n_estimators': 300,
    'max_samples': 0.9,
    'max_features': 0.9,
    'sampling_strategy': 'auto',
    'replacement': True,
    'random_state': 42,
    'n_jobs': -1
}

# EASY ENSEMBLE CONFIGURATION
EASY_ENSEMBLE_CONFIG = {
    'n_estimators': 100,
    'random_state': 42,
    'n_jobs': -1
}

# GB-LGBM HYBRID CONFIGURATION (original)
GB_LGBM_HYBRID_CONFIG = {
    'gb_config': {
        'n_estimators': 300,
        'learning_rate': 0.1,
        'max_depth': 5,
        'min_samples_split': 5,
        'min_samples_leaf': 2,
        'subsample': 0.8,
        'random_state': 42,
    },
    'lgbm_config': {
        'learning_rate': 0.03,
        'max_depth': 40,
        'min_data_in_leaf': 10,
        'n_estimators': 500,
        'num_leaves': 50,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 3,
        'objective': 'binary',
        'metric': 'auc',
        'random_state': 42,
        'verbosity': -1,
    },
    'voting_config': {
        'voting': 'soft',
        'weights': [1, 1],
        'n_jobs': -1,
    },
}

# Master dictionary containing all model configs
ALL_MODEL_CONFIGS = {
    'GB-LGBM-Hybrid': GB_LGBM_HYBRID_CONFIG,
    'GradientBoosting': GB_CONFIG,
    'LightGBM': LGBM_CONFIG,
    'XGBoost': XGB_CONFIG,
    'RandomForest': RF_CONFIG,
    'CatBoost': CATBOOST_CONFIG,
    'MLP': MLP_CONFIG,
    'AdaBoost': ADA_CONFIG,
    'KNN': KNN_CONFIG,
    'LogisticRegression': LR_CONFIG,
    'BalancedBagging': BALANCED_BAGGING_CONFIG,
    'EasyEnsemble': EASY_ENSEMBLE_CONFIG,
}

# List of models to compare (order matters for printing)
MODELS_TO_COMPARE = [
    'GB-LGBM-Hybrid',
    'GradientBoosting',
    'LightGBM',
    'XGBoost',
    'RandomForest',
    'CatBoost',
    'MLP',
    'AdaBoost',
    'KNN',
    'LogisticRegression',
    'BalancedBagging',
    'EasyEnsemble'
]

SELECTION_CRITERIA = {
    'min_precision_threshold': 0.60,
    'min_samples_per_class': 0,
    'min_total_samples': 0,
}

# ---------- Number of CV repeats ----------
N_REPEATS = 5   

print("\n" + "="*80)
print("VALIDATION CONFIGURATION")
print("="*80)
print(f"Optimal window length : {WINDOW_LENGTH} frames")
print(f"Best start frames     : Phase1=12, Phase2=105, Phase3=146, Phase4=233")
print(f"Balance ratio         : 10:1  (Normal : iPS)")
print(f"CV repeats            : {N_REPEATS}")
print(f"Models to compare     : {len(MODELS_TO_COMPARE)}")
for m in MODELS_TO_COMPARE:
    print(f"  - {m}")
print("="*80)

In [ ]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def frame_to_time(frame_number):
    """Convert frame index to experimental time string."""
    phase_timing = {
        1: {'start_frame': 0,   'end_frame': 101, 'start_min': 0},
        2: {'start_frame': 102, 'end_frame': 142, 'start_min': 18*60},
        3: {'start_frame': 143, 'end_frame': 232, 'start_min': 25*60+50},
        4: {'start_frame': 233, 'end_frame': 274, 'start_min': 41*60+50},
    }
    for phase, info in phase_timing.items():
        if info['start_frame'] <= frame_number <= info['end_frame']:
            frames_in  = frame_number - info['start_frame']
            total_min  = info['start_min'] + frames_in * 10
            h, m       = divmod(total_min, 60)
            return f"{h}h{m:02d}min" if m else f"{h}h"
    return f"Frame {frame_number}"


def get_phase_for_frame(frame):
    for phase, info in PHASES.items():
        if info['frames'][0] <= frame <= info['frames'][1]:
            return phase
    return None


def setup_publication_fonts():
    """Return best available serif font for publication figures."""
    available = {f.name for f in fm.fontManager.ttflist}
    for candidate in ['Times New Roman', 'Liberation Serif',
                      'DejaVu Serif', 'Times', 'Georgia']:
        if candidate in available:
            return candidate
    return 'serif'


FONT_FAMILY = setup_publication_fonts()

# IMPROVED: Better RC_PARAMS for publication-quality figures
RC_PARAMS = {
    'font.family':       FONT_FAMILY,
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.labelsize':    11,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'figure.dpi':        300,
    'savefig.dpi':       300,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.25,
    'grid.linestyle':    '--',
    'axes.linewidth':    1.2,
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
}

PHASE_COLORS  = {1: '#2166AC', 2: '#4DAC26', 3: '#D7191C', 4: '#7B2D8B'}
METRIC_COLORS = {
    'precision':          '#2166AC',
    'recall':             '#4DAC26',
    'f1':                 '#D7191C',
    'balanced_accuracy':  '#FF7F00',
    'mcc':                '#7B2D8B',
    'roc_auc':            '#A6611A',
    'pr_auc':             '#018571',
    'sensitivity':        '#80CDC1',
    'specificity':        '#DFC27D',
}

METRIC_LABELS = {
    'precision':          'Precision',
    'recall':             'Recall',
    'f1':                 'F1-Score',
    'balanced_accuracy':  'Balanced Accuracy',
    'mcc':                'MCC',
    'roc_auc':            'ROC-AUC',
    'pr_auc':             'PR-AUC',
    'sensitivity':        'Sensitivity',
    'specificity':        'Specificity',
}

print(f"Publication font : {FONT_FAMILY}")

In [ ]:
# =============================================================================
# CELL 4: DATA LOADER
# =============================================================================

class DataLoader:
    """Load and validate all FOV data (identical to reference)."""

    def __init__(self, consensus_path=CONSENSUS_PATH, verbose=True):
        self.consensus_path = Path(consensus_path)
        self.verbose = verbose
        self.data_stats = {}

    def validate_data_structure(self):
        print("Validating data structure and finding FOVs...")
        if not self.consensus_path.exists():
            raise FileNotFoundError(
                f"Consensus directory not found: {self.consensus_path}")

        all_dirs   = [d for d in self.consensus_path.iterdir() if d.is_dir()]
        valid_fovs = []

        for fov_dir in all_dirs:
            lf = fov_dir / "Labelled" / "consensus_retrospective_labels.csv"
            if lf.exists():
                try:
                    t = pd.read_csv(lf)
                    if len(t) > 0 and 'Label' in t.columns:
                        valid_fovs.append(fov_dir.name)
                except Exception:
                    pass

        if not valid_fovs:
            raise ValueError("No valid FOVs found.")

        valid_fovs.sort(key=lambda x: (0, int(x)) if x.isdigit() else (1, x))
        print(f"   Total dirs : {len(all_dirs)}  |  Valid FOVs : {len(valid_fovs)} → {valid_fovs}")
        return valid_fovs

    def load_single_fov(self, fov_dir):
        fov_num = fov_dir.name
        lf = fov_dir / "Labelled" / "consensus_retrospective_labels.csv"
        try:
            df = pd.read_csv(lf)
            df['FOV'] = fov_num
            if df.empty or 'Label' not in df.columns:
                return None
            cell_lbl = df.groupby('Cell ID')['Label'].first()
            if self.verbose:
                n = (cell_lbl == 0).sum()
                i = (cell_lbl == 1).sum()
                print(f"   FOV {fov_num}: {len(df):,} records, "
                      f"{len(cell_lbl)} cells (Normal:{n}, iPS:{i})")
            return df
        except Exception as e:
            print(f"Error loading FOV {fov_num}: {e}")
            return None

    def load_all_data(self):
        print("\nLoading all FOV data …")
        valid_fovs = self.validate_data_structure()
        all_data   = []
        for fov_name in sorted(
                valid_fovs,
                key=lambda x: int(x) if x.isdigit() else float('inf')):
            df = self.load_single_fov(self.consensus_path / fov_name)
            if df is not None:
                all_data.append(df)

        if not all_data:
            raise ValueError("No data loaded.")

        combined = pd.concat(all_data, ignore_index=True)
        uc = combined.groupby(['FOV', 'Cell ID'])['Label'].first()
        tc, nc, ic = len(uc), (uc==0).sum(), (uc==1).sum()
        print(f"\nData Loading Summary:")
        print(f"   Total records : {len(combined):,}")
        print(f"   Total cells   : {tc:,}  (Normal: {nc:,}, iPS: {ic:,})")
        print(f"   iPS rate      : {ic/tc*100:.2f}%")
        print(f"   Frame range   : {combined['Frame'].min()}–{combined['Frame'].max()}")
        return combined


print("\n" + "="*80)
print("LOADING DATA")
print("="*80)
data_loader = DataLoader()
df_full     = data_loader.load_all_data()
print(f"\nData shape: {df_full.shape}")

In [ ]:
# =============================================================================
# CELL 5: FEATURE PROCESSOR WITH SURVIVAL-BASED METHODOLOGY
# =============================================================================

class FeatureProcessor:
    """
    Feature processing with survival-based cell inclusion strategy
    
    METHODOLOGY:
    1. Start with balanced dataset at Length 5 ONLY
    2. For longer lengths (7,9,11,13,15,17, etc.), use ONLY cells that survived from Length 5
    3. NO fresh balancing at longer lengths - just evaluate surviving cells
    """

    def __init__(self, feature_categories=FEATURE_CATEGORIES, balance_ratio=(10, 1)):
        self.feature_categories = feature_categories
        self.all_features = ALL_FEATURES
        self.scalers = {}
        self.feature_stats = {}
        self.outlier_bounds = {}
        self.balance_ratio = balance_ratio
        self.selected_cells_by_phase = {}
        self.balanced_phase_data = {}
        self.length5_baseline_cells = {}
        self.df = None  # stored by precompute()

    # -------------------------------------------------------------------------
    # Added for CV file: store df once so methods below need no df argument
    # -------------------------------------------------------------------------
    def precompute(self, df):
        self.df = df
        print("Feature processor ready.")

    # -------------------------------------------------------------------------
    # Reference methods — unchanged
    # -------------------------------------------------------------------------

    def validate_features(self, df):
        """Validate and clean features"""
        available_features = [f for f in self.all_features if f in df.columns]
        return available_features

    def get_phase_for_frame(self, frame):
        """Get phase number for a given frame"""
        for phase, info in PHASES.items():
            if info['frames'][0] <= frame <= info['frames'][1]:
                return phase
        return None

    def identify_daughter_cells(self, df):
        """Identify daughter cells using Parent ID and Daughter IDs columns"""
        daughter_cells = set()
        if 'Parent ID' in df.columns:
            daughters = df[df['Parent ID'] != 0][['FOV', 'Cell ID']].drop_duplicates()
            for _, row in daughters.iterrows():
                daughter_cells.add((row['FOV'], row['Cell ID']))
        return daughter_cells

    def get_cell_birth_and_death_frames(self, df):
        """Get birth and death frames for each cell"""
        cell_info = {}
        for (fov, cell_id), group in df.groupby(['FOV', 'Cell ID']):
            birth_frame = group['Frame'].min()
            death_frame = group['Frame'].max()
            cell_info[(fov, cell_id)] = {
                'birth': birth_frame,
                'death': death_frame
            }
        return cell_info

    def get_cells_for_start_frame_window(self, df, start_frame, window_length):
        """
        Cell inclusion strategy for Length 5 baseline creation:
        1. Include ONLY cells that were already born AT OR BEFORE start_frame
        2. EXCLUDE cells that died before the window end
        3. EXCLUDE daughter cells (Parent ID != 0)
        """
        window_end_frame = start_frame + window_length - 1
        daughter_cells = self.identify_daughter_cells(df)
        cell_info = self.get_cell_birth_and_death_frames(df)
        
        qualifying_cells = []
        
        for (fov, cell_id), info in cell_info.items():
            birth = info['birth']
            death = info['death']
            
            if (fov, cell_id) in daughter_cells:
                continue
            if birth > start_frame:
                continue
            if death < window_end_frame:
                continue
            if birth <= start_frame and death >= start_frame:
                qualifying_cells.append((fov, cell_id))
        
        return qualifying_cells

    def create_length5_baseline(self, df, start_frame, normalize=True, normalization_method='robust'):
        """Create balanced Length 5 baseline dataset - ONLY place where balancing occurs"""
        window_length = 5
        window_end_frame = start_frame + window_length - 1
        
        qualifying_cells = self.get_cells_for_start_frame_window(df, start_frame, window_length)
        
        if len(qualifying_cells) == 0:
            return None
        
        window_data = df[
            (df['Frame'] >= start_frame) & 
            (df['Frame'] <= window_end_frame)
        ].copy()
        
        cell_mask = window_data.set_index(['FOV', 'Cell ID']).index.isin(qualifying_cells)
        window_data = window_data[cell_mask].reset_index(drop=True)
        
        if len(window_data) == 0:
            return None
        
        cell_labels = window_data.groupby(['FOV', 'Cell ID'])['Label'].first()
        normal_cells = (cell_labels == 0).sum()
        ips_cells = (cell_labels == 1).sum()
        
        if ips_cells == 0:
            return None
        
        if normal_cells > ips_cells * 10:
            target_normal = ips_cells * 10
            
            normal_cell_ids = [
                (fov, cid) for (fov, cid), label in cell_labels.items() 
                if label == 0
            ]
            
            rng = np.random.RandomState(42)
            selected_indices = rng.choice(
                len(normal_cell_ids), 
                size=target_normal, 
                replace=False
            )
            selected_normal_ids = [normal_cell_ids[i] for i in selected_indices]
            
            ips_cell_ids = [
                (fov, cid) for (fov, cid), label in cell_labels.items() 
                if label == 1
            ]
            
            baseline_cell_ids = selected_normal_ids + ips_cell_ids
        else:
            baseline_cell_ids = list(cell_labels.index)
        
        self.length5_baseline_cells[start_frame] = baseline_cell_ids
        return baseline_cell_ids

    def create_survival_window_data(self, df, start_frame, window_length, normalize=True,
                                  normalization_method='robust'):
        """Create dataset using survival from Length 5 baseline"""
        window_end_frame = start_frame + window_length - 1
        phase = self.get_phase_for_frame(start_frame)
        
        if window_length == 5:
            baseline_cells = self.create_length5_baseline(df, start_frame, normalize, normalization_method)
            if baseline_cells is None:
                return None
            selected_cell_ids = baseline_cells
            baseline_count = len(baseline_cells)
        else:
            # Create WL=5 baseline on demand if not yet computed
            if start_frame not in self.length5_baseline_cells:
                baseline_cells = self.create_length5_baseline(df, start_frame, normalize, normalization_method)
                if baseline_cells is None:
                    return None
            
            baseline_cells = self.length5_baseline_cells[start_frame]
            baseline_count = len(baseline_cells)
            cell_info = self.get_cell_birth_and_death_frames(df)
            
            surviving_cells = []
            for cell_id in baseline_cells:
                if cell_id in cell_info:
                    death_frame = cell_info[cell_id]['death']
                    if death_frame >= window_end_frame:
                        surviving_cells.append(cell_id)
            
            selected_cell_ids = surviving_cells
        
        if len(selected_cell_ids) == 0:
            return None
        
        window_data = df[
            (df['Frame'] >= start_frame) & 
            (df['Frame'] <= window_end_frame)
        ].copy()
        
        cell_mask = window_data.set_index(['FOV', 'Cell ID']).index.isin(selected_cell_ids)
        window_data = window_data[cell_mask].reset_index(drop=True)
        
        if len(window_data) == 0:
            return None
        
        cell_labels = window_data.groupby(['FOV', 'Cell ID'])['Label'].first()
        normal_cells = (cell_labels == 0).sum()
        ips_cells = (cell_labels == 1).sum()
        total_cells = len(cell_labels)
        
        if ips_cells == 0:
            return None
        
        available_features = self.validate_features(window_data)
        if not available_features:
            return None
        
        window_data = self.comprehensive_outlier_detection(window_data, available_features)
        window_data = self.normalize_features(window_data, available_features, method=normalization_method)
        
        excluded_cols = ['FOV', 'Cell ID', 'Frame', 'Label', 'Parent ID', 'Daughter IDs', 
                        'Division Frame', 'X', 'Y']
        final_features = [
            col for col in window_data.columns 
            if col in available_features or any(feat in col for feat in available_features)
        ]
        final_features = [col for col in final_features if col not in excluded_cols]
        
        if len(final_features) == 0:
            return None
        
        X = window_data[final_features].values
        y = window_data['Label'].values
        
        if len(X) == 0:
            return None
        
        unique_classes, class_counts = np.unique(y, return_counts=True)
        class_dist = dict(zip(unique_classes, class_counts))
        
        window_info = {
            'start_frame': start_frame,
            'window_length': window_length,
            'end_frame': window_end_frame,
            'phase': phase,
            'n_samples': len(X),
            'n_features': X.shape[1],
            'features': final_features,
            'class_distribution': class_dist,
            'balance_ratio': normal_cells / ips_cells if ips_cells > 0 else 0,
            'X': X,
            'y': y,
            'total_cells': total_cells,
            'normal_cells': normal_cells,
            'ips_cells': ips_cells,
            'survival_methodology_used': True,
            'baseline_cell_count': baseline_count,
            'qualifying_cells_count': len(selected_cell_ids),
            'X_raw': window_data[final_features].values,  # for independent test scaler
        }
        
        return window_info

    def create_time_window_data(self, start_frame, window_length):
        """Main interface using self.df (stored by precompute)"""
        return self.create_survival_window_data(self.df, start_frame, window_length,
                                                normalize=True,
                                                normalization_method='robust')

    def comprehensive_outlier_detection(self, df, features):
        """Conservative outlier detection using percentile capping"""
        df_clean = df.copy()
        for feature in features:
            if feature not in df.columns:
                continue
            feature_data = df[feature].dropna()
            if len(feature_data) == 0:
                continue
            lower_bound = np.percentile(feature_data, 1)
            upper_bound = np.percentile(feature_data, 99)
            df_clean[feature] = np.clip(df_clean[feature], lower_bound, upper_bound)
        return df_clean

    def normalize_features(self, df, features, method='robust'):
        """Normalize features using RobustScaler"""
        normalized_df = df.copy()
        for feature in features:
            if feature not in df.columns:
                continue
            feature_data = df[feature].values.reshape(-1, 1)
            scaler = RobustScaler()
            normalized_data = scaler.fit_transform(feature_data)
            normalized_df[feature] = normalized_data.flatten()
            self.scalers[feature] = scaler
        return normalized_df

    # -------------------------------------------------------------------------
    # Added for CV file: holdout / independent test need external scaler params
    # -------------------------------------------------------------------------
    def create_time_window_data_with_train_scaler(self, start_frame, window_length,
                                                   train_scaler_params):
        """
        Same survival methodology but applies scaler params fitted on training data.
        Used by holdout and independent test experiments.
        """
        df = self.df
        window_end_frame = start_frame + window_length - 1
        phase = self.get_phase_for_frame(start_frame)

        if window_length == 5:
            baseline_cells = self.create_length5_baseline(df, start_frame)
            if baseline_cells is None:
                return None
            selected_cell_ids = baseline_cells
            baseline_count = len(baseline_cells)
        else:
            if start_frame not in self.length5_baseline_cells:
                baseline_cells = self.create_length5_baseline(df, start_frame)
                if baseline_cells is None:
                    return None
            baseline_cells = self.length5_baseline_cells[start_frame]
            baseline_count = len(baseline_cells)
            cell_info = self.get_cell_birth_and_death_frames(df)

            surviving_cells = []
            for cell_id in baseline_cells:
                if cell_id in cell_info:
                    if cell_info[cell_id]['death'] >= window_end_frame:
                        surviving_cells.append(cell_id)
            selected_cell_ids = surviving_cells

        if len(selected_cell_ids) == 0:
            return None

        window_data = df[
            (df['Frame'] >= start_frame) &
            (df['Frame'] <= window_end_frame)
        ].copy()

        cell_mask = window_data.set_index(['FOV', 'Cell ID']).index.isin(selected_cell_ids)
        window_data = window_data[cell_mask].reset_index(drop=True)

        if len(window_data) == 0:
            return None

        cell_labels = window_data.groupby(['FOV', 'Cell ID'])['Label'].first()
        normal_cells = (cell_labels == 0).sum()
        ips_cells    = (cell_labels == 1).sum()

        if ips_cells == 0:
            return None

        available_features = self.validate_features(window_data)
        if not available_features:
            return None

        # Outlier capping then scale using training scaler params
        window_data = self.comprehensive_outlier_detection(window_data, available_features)

        lo  = train_scaler_params['lo']
        hi  = train_scaler_params['hi']
        med = train_scaler_params['med']
        iqr = train_scaler_params['iqr']

        X_raw = window_data[available_features].values.astype(np.float64)
        X = np.clip(X_raw, lo, hi)
        X = (X - med) / iqr

        y = window_data['Label'].values

        unique_classes, class_counts = np.unique(y, return_counts=True)
        class_dist = dict(zip(unique_classes.tolist(), class_counts.tolist()))

        return {
            'start_frame': start_frame,
            'window_length': window_length,
            'end_frame': window_end_frame,
            'phase': phase,
            'n_samples': len(X),
            'n_features': X.shape[1],
            'features': available_features,
            'class_distribution': class_dist,
            'X': X,
            'y': y,
            'total_cells': len(cell_labels),
            'normal_cells': normal_cells,
            'ips_cells': ips_cells,
            'balance_ratio': normal_cells / ips_cells if ips_cells > 0 else 0,
            'survival_methodology_used': True,
            'baseline_cell_count': baseline_count,
            'qualifying_cells_count': len(selected_cell_ids),
        }

    @staticmethod
    def print_dataset_summary(window_data, label="Dataset"):
        if window_data is None:
            print(f"   {label}: None")
            return
        nc       = window_data['normal_cells']
        ic       = window_data['ips_cells']
        tot      = window_data['total_cells']
        rt       = window_data['balance_ratio']
        survival = window_data.get('survival_methodology_used', False)
        baseline = window_data.get('baseline_cell_count', 'N/A')

        print(f"   {label}:")
        print(f"      Samples      : {window_data['n_samples']}")
        print(f"      Total cells  : {tot} (Normal: {nc}, iPS: {ic})")
        print(f"      Ratio        : {rt:.1f}:1")
        print(f"      Class dist   : {window_data['class_distribution']}")
        if survival:
            print(f"      Methodology  : Survival-based (baseline: {baseline} cells)")


print("\n" + "="*80)
print("INITIALISING FEATURE PROCESSOR — SURVIVAL-BASED METHODOLOGY")
print("="*80)
print("Methodology:")
print("  - Length 5: Global random balanced baseline (10:1 ratio)")
print("  - Length 7+: Only surviving cells from Length 5 baseline")
print("  - NO fresh balancing at longer window lengths")
print("="*80)
feature_processor = FeatureProcessor(balance_ratio=(10, 1))
feature_processor.precompute(df_full)
print("="*80)

In [ ]:
# =============================================================================
# CELL 6: MODEL FACTORY WITH MULTIPLE MODELS 
# =============================================================================

# Import additional models
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

# Try importing optional libraries
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available")

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM not available")

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("CatBoost not available")

try:
    from imblearn.ensemble import BalancedBaggingClassifier, EasyEnsembleClassifier
    IMBLEARN_AVAILABLE = True
    print("imbalanced-learn available")
except ImportError:
    IMBLEARN_AVAILABLE = False
    print("imbalanced-learn not available")


class ModelFactory:
    """Create multiple classification models for comparison."""
    
    def __init__(self):
        # Reference to global model configs
        self.xgb_config = XGB_CONFIG
        self.lgbm_config = LGBM_CONFIG
        self.rf_config = RF_CONFIG
        self.gb_config = GB_CONFIG
        self.catboost_config = CATBOOST_CONFIG
        self.mlp_config = MLP_CONFIG
        self.ada_config = ADA_CONFIG
        self.knn_config = KNN_CONFIG
        self.lr_config = LR_CONFIG
        self.balanced_bagging_config = BALANCED_BAGGING_CONFIG
        self.easy_ensemble_config = EASY_ENSEMBLE_CONFIG
        self.gb_lgbm_hybrid_config = GB_LGBM_HYBRID_CONFIG

    def create_model(self, model_name, random_state=42):
        """Create a model by name with the specified random state."""
        
        # GB-LGBM Hybrid (original)
        if model_name == 'GB-LGBM-Hybrid':
            return self.create_gb_lgbm_hybrid(random_state)
        
        # Gradient Boosting
        elif model_name == 'GradientBoosting':
            config = self.gb_config.copy()
            config['random_state'] = random_state
            return GradientBoostingClassifier(**config)
        
        # LightGBM
        elif model_name == 'LightGBM':
            if not LIGHTGBM_AVAILABLE:
                print(f"Warning: LightGBM not available for {model_name}")
                return None
            config = self.lgbm_config.copy()
            config['random_state'] = random_state
            return LGBMClassifier(**config)
        
        # XGBoost
        elif model_name == 'XGBoost':
            if not XGBOOST_AVAILABLE:
                print(f"Warning: XGBoost not available for {model_name}")
                return None
            config = self.xgb_config.copy()
            config['random_state'] = random_state
            return XGBClassifier(**config)
        
        # Random Forest
        elif model_name == 'RandomForest':
            config = self.rf_config.copy()
            config['random_state'] = random_state
            return RandomForestClassifier(**config)
        
        # CatBoost
        elif model_name == 'CatBoost':
            if not CATBOOST_AVAILABLE:
                print(f"Warning: CatBoost not available for {model_name}")
                return None
            config = self.catboost_config.copy()
            config['random_seed'] = random_state
            return CatBoostClassifier(**config)
        
        # MLP Neural Network
        elif model_name == 'MLP':
            config = self.mlp_config.copy()
            config['random_state'] = random_state
            return MLPClassifier(**config)
        
        # AdaBoost
        elif model_name == 'AdaBoost':
            config = self.ada_config.copy()
            config['random_state'] = random_state
            return AdaBoostClassifier(**config)
        
        # K-Nearest Neighbors
        elif model_name == 'KNN':
            return KNeighborsClassifier(**self.knn_config)
        
        # Logistic Regression
        elif model_name == 'LogisticRegression':
            config = self.lr_config.copy()
            config['random_state'] = random_state
            return LogisticRegression(**config)
        
        # Balanced Bagging -  (base_clf properly defined)
        elif model_name == 'BalancedBagging':
            if not IMBLEARN_AVAILABLE:
                print(f"Warning: imbalanced-learn not available for {model_name}")
                return None
            config = self.balanced_bagging_config.copy()
            config['random_state'] = random_state
            
            # Define base classifier for BalancedBagging
            base_clf = RandomForestClassifier(n_estimators=50, random_state=random_state)
            
            return BalancedBaggingClassifier(
                estimator=base_clf, **config
            )
        
        # Easy Ensemble -  (base_clf properly defined)
        elif model_name == 'EasyEnsemble':
            if not IMBLEARN_AVAILABLE:
                print(f"Warning: imbalanced-learn not available for {model_name}")
                return None
            config = self.easy_ensemble_config.copy()
            config['random_state'] = random_state
            
            # Define base classifier for EasyEnsemble
            base_clf = RandomForestClassifier(n_estimators=50, random_state=random_state)
            
            return EasyEnsembleClassifier(
                estimator=base_clf, **config
            )
        
        else:
            raise ValueError(f"Unknown model: {model_name}")
    
    def create_gb_lgbm_hybrid(self, random_state=42):
        """Create GB-LGBM hybrid model with specified random state."""
        cfg = self.gb_lgbm_hybrid_config
        
        # Create copies of configs to avoid modifying the original
        gb_config = cfg['gb_config'].copy()
        lgbm_config = cfg['lgbm_config'].copy()
        
        # Override random states with the provided seed
        gb_config['random_state'] = random_state
        lgbm_config['random_state'] = random_state
        
        gb = GradientBoostingClassifier(**gb_config)
        
        if not LIGHTGBM_AVAILABLE:
            print("Warning: LightGBM not available, returning GB only.")
            return gb
            
        lgbm = LGBMClassifier(**lgbm_config)
        vc = cfg['voting_config']
        
        return VotingClassifier(
            estimators=[('gb', gb), ('lgbm', lgbm)],
            voting=vc['voting'],
            weights=vc['weights'],
            n_jobs=vc['n_jobs'],
        )
    
    def get_available_models(self):
        """Return list of models that are available with current libraries."""
        available = []
        for model_name in MODELS_TO_COMPARE:
            if model_name == 'GB-LGBM-Hybrid':
                available.append(model_name)
            elif model_name == 'LightGBM' and not LIGHTGBM_AVAILABLE:
                continue
            elif model_name == 'XGBoost' and not XGBOOST_AVAILABLE:
                continue
            elif model_name == 'CatBoost' and not CATBOOST_AVAILABLE:
                continue
            elif model_name in ['BalancedBagging', 'EasyEnsemble'] and not IMBLEARN_AVAILABLE:
                continue
            else:
                available.append(model_name)
        return available


# Initialize model factory
model_factory = ModelFactory()

print("\n" + "="*80)
print("MODEL FACTORY INITIALIZED")
print("="*80)
available_models = model_factory.get_available_models()
print(f"Available models: {len(available_models)}/{len(MODELS_TO_COMPARE)}")
for m in available_models:
    print(f"  - {m}")
print("="*80)

In [ ]:
# =============================================================================
# CELL 7: EVALUATION ENGINE
# =============================================================================

from sklearn.model_selection import KFold

# Number of bootstrap resamples used when evaluating on a fixed test set.
N_BOOTSTRAP_TEST = 20

class EvaluationEngine:
    """Evaluation engine that uses survival-based datasets without additional balancing"""

    def __init__(self, model_factory, feature_processor):
        self.model_factory = model_factory
        self.feature_processor = feature_processor
        self.results = []

    def find_optimal_threshold(self, y_true, y_prob):
        """Find optimal threshold using precision-recall curve to maximize F1"""
        if len(np.unique(y_true)) < 2:
            return 0.5

        try:
            precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
            f1_scores = []
            for p, r in zip(precision, recall):
                if p + r > 0:
                    f1_scores.append(2 * (p * r) / (p + r))
                else:
                    f1_scores.append(0)
            best_idx = np.argmax(f1_scores)
            best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
            return best_threshold
        except Exception:
            return 0.5

    def calculate_comprehensive_metrics(self, y_true, y_pred, y_prob=None):
        """Calculate comprehensive evaluation metrics"""
        metrics = {}
        metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
        metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
        metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
        metrics['f05'] = fbeta_score(y_true, y_pred, beta=0.5, zero_division=0)
        metrics['f2'] = fbeta_score(y_true, y_pred, beta=2.0, zero_division=0)
        metrics['balanced_accuracy'] = balanced_accuracy_score(y_true, y_pred)
        metrics['accuracy'] = accuracy_score(y_true, y_pred)

        if y_prob is not None and len(np.unique(y_true)) == 2:
            try:
                metrics['roc_auc'] = roc_auc_score(y_true, y_prob)
                metrics['pr_auc'] = average_precision_score(y_true, y_prob)
            except:
                metrics['roc_auc'] = 0.0
                metrics['pr_auc'] = 0.0
        else:
            metrics['roc_auc'] = 0.0
            metrics['pr_auc'] = 0.0

        metrics['mcc'] = matthews_corrcoef(y_true, y_pred)

        if len(np.unique(y_true)) == 2:
            cm = confusion_matrix(y_true, y_pred)
            if cm.shape == (2, 2):
                tn, fp, fn, tp = cm.ravel()
                sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
                specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
                metrics['sensitivity'] = sensitivity
                metrics['specificity'] = specificity
                metrics['geometric_mean'] = np.sqrt(sensitivity * specificity)
                metrics['true_positive'] = int(tp)
                metrics['false_positive'] = int(fp)
                metrics['true_negative'] = int(tn)
                metrics['false_negative'] = int(fn)
            else:
                metrics['sensitivity'] = 0.0
                metrics['specificity'] = 0.0
                metrics['geometric_mean'] = 0.0
                metrics['true_positive'] = 0
                metrics['false_positive'] = 0
                metrics['true_negative'] = 0
                metrics['false_negative'] = 0
        else:
            metrics['sensitivity'] = 0.0
            metrics['specificity'] = 0.0
            metrics['geometric_mean'] = 0.0
            metrics['true_positive'] = 0
            metrics['false_positive'] = 0
            metrics['true_negative'] = 0
            metrics['false_negative'] = 0

        return metrics

    def evaluate_single_model(self, X, y, model_name, model, cv_folds=5, cv_repeats=5):
        """Evaluate GB-LGBM hybrid with cross-validation preserving imbalanced data"""
        all_fold_metrics = []

        for repeat in range(cv_repeats):
            cv = KFold(n_splits=cv_folds, shuffle=True, random_state=42 + repeat)

            for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
                X_train, X_val = X[train_idx], X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]

                try:
                    model_copy = self.model_factory.create_gb_lgbm_hybrid()
                    model_copy.fit(X_train, y_train)

                    y_prob = model_copy.predict_proba(X_val)[:, 1]

                    optimal_threshold = self.find_optimal_threshold(y_val, y_prob)
                    y_pred = (y_prob >= optimal_threshold).astype(int)

                    metrics = self.calculate_comprehensive_metrics(y_val, y_pred, y_prob)
                    metrics['fold'] = fold
                    metrics['repeat'] = repeat
                    metrics['optimal_threshold'] = optimal_threshold

                    all_fold_metrics.append(metrics)

                except Exception as e:
                    continue

        if not all_fold_metrics:
            return None

        aggregated_metrics = {}
        metric_names = [key for key in all_fold_metrics[0].keys()
                        if key not in ['fold', 'repeat', 'optimal_threshold',
                                       'true_positive', 'false_positive',
                                       'true_negative', 'false_negative']]

        for metric in metric_names:
            values = [fm[metric] for fm in all_fold_metrics
                      if metric in fm and not np.isnan(fm[metric])]
            if values:
                aggregated_metrics[f'{metric}_mean'] = np.mean(values)
                aggregated_metrics[f'{metric}_std'] = np.std(values)

        aggregated_metrics['model_type'] = model_name
        aggregated_metrics['n_evaluations'] = len(all_fold_metrics)

        return aggregated_metrics, all_fold_metrics

    def run_cross_validation(self, X, y, model_name, n_folds=5, n_repeats=5):
        """
        Cross-validation wrapper for all models.
        Uses KFold to preserve natural class distribution.
        GB-LGBM-Hybrid always uses random_state=42 across folds, consistent with
        the time-window selection reference.
        """
        all_fold_metrics = []

        for repeat in range(n_repeats):
            cv = KFold(n_splits=n_folds, shuffle=True, random_state=42 + repeat)

            for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
                X_train, X_val = X[train_idx], X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]

                try:
                    if model_name == 'GB-LGBM-Hybrid':
                        model = self.model_factory.create_gb_lgbm_hybrid()
                    else:
                        model = self.model_factory.create_model(
                            model_name,
                            random_state=42 + repeat * 10 + fold
                        )
                    if model is None:
                        continue

                    model.fit(X_train, y_train)
                    y_prob = model.predict_proba(X_val)[:, 1]
                    thr = self.find_optimal_threshold(y_val, y_prob)
                    y_pred = (y_prob >= thr).astype(int)

                    fm = self.calculate_comprehensive_metrics(y_val, y_pred, y_prob)
                    fm['fold'] = fold
                    fm['repeat'] = repeat
                    all_fold_metrics.append(fm)

                except Exception as e:
                    print(f"     {model_name} Fold {fold} rep {repeat} failed: {e}")
                    continue

        return self.aggregate_fold_metrics(all_fold_metrics), all_fold_metrics

    def aggregate_fold_metrics(self, fold_metrics_list):
        """Aggregate metrics across folds"""
        if not fold_metrics_list:
            return {}

        scalar_keys = [k for k in fold_metrics_list[0].keys()
                       if isinstance(fold_metrics_list[0][k], (float, np.floating))]

        agg = {}
        for k in scalar_keys:
            vals = [fm[k] for fm in fold_metrics_list if not np.isnan(fm[k])]
            if vals:
                agg[f'{k}_mean'] = float(np.mean(vals))
                agg[f'{k}_std'] = float(np.std(vals))

        agg['n_evaluations'] = len(fold_metrics_list)
        return agg

    def evaluate_on_test_set(self, X_train, y_train, X_test, y_test,
                              model_name, n_repeats=5):
        """Evaluate a model on a fixed test set."""
        all_metrics = []
        n_test = len(X_test)

        for rep in range(n_repeats):
            try:
                model = self.model_factory.create_model(
                    model_name,
                    random_state=42 + rep
                )
                if model is None:
                    continue

                model.fit(X_train, y_train)
                y_prob_full = model.predict_proba(X_test)[:, 1]

                # Bootstrap resample the test set to obtain SD
                for b in range(N_BOOTSTRAP_TEST):
                    rng = np.random.RandomState(42 + rep * 1000 + b)
                    idx = rng.choice(n_test, size=n_test, replace=True)
                    y_b = y_test[idx]
                    p_b = y_prob_full[idx]

                    if len(np.unique(y_b)) < 2:
                        continue

                    thr = self.find_optimal_threshold(y_b, p_b)
                    y_pred = (p_b >= thr).astype(int)
                    fm = self.calculate_comprehensive_metrics(y_b, y_pred, p_b)
                    fm['repeat'] = rep
                    fm['bootstrap'] = b
                    all_metrics.append(fm)

            except Exception as e:
                print(f"     {model_name} Repeat {rep} failed: {e}")
                continue

        return self.aggregate_fold_metrics(all_metrics), all_metrics


# Initialize evaluation engine
evaluation_engine = EvaluationEngine(model_factory, feature_processor)

print("="*80)
print("EVALUATION ENGINE READY")
print("="*80)
print(f"  CV Method    : KFold (preserves natural class distribution)")
print(f"  CV folds     : 5 x N_REPEATS")
print(f"  Holdout      : N_REPEATS models x {N_BOOTSTRAP_TEST} bootstrap draws")
print(f"  Threshold    : Optimal F1 from precision-recall curve")
print(f"  Models ready : {len(model_factory.get_available_models())}")
print("="*80)

In [ ]:
# =============================================================================
# CELL 8: CROSS-VALIDATION EXPERIMENT
# =============================================================================

def perform_cross_validation(model_name, n_repeats=N_REPEATS):
    """5-fold CV on the optimal window of each phase using survival-based methodology."""
    print("\n" + "="*80)
    print(f"EXPERIMENT 1: CROSS-VALIDATION - {model_name}")
    print("="*80)
    print(f"5-fold CV × {n_repeats} repeat(s) = {5*n_repeats} evaluations per phase")
    print(f"Preprocessing : Survival-based methodology (Length 5 baseline + surviving cells)")
    print("="*80)

    cv_results = {}

    for phase in [1, 2, 3, 4]:
        cfg          = BEST_CONFIGS[phase]
        start_frame  = cfg['start_frame']
        wl           = cfg['window_length']
        end_frame    = start_frame + wl - 1

        print(f"\nPhase {phase}: Start Frame {start_frame}, "
              f"Window Length {wl} (frames {start_frame}–{end_frame})")
        print("-"*60)

        win = feature_processor.create_time_window_data(start_frame, wl)
        if win is None:
            print(f" Failed to create dataset for Phase {phase}")
            continue

        feature_processor.print_dataset_summary(win, "Dataset")

        X, y = win['X'], win['y']
        agg, fold_list = evaluation_engine.run_cross_validation(
            X, y, model_name, n_folds=5, n_repeats=n_repeats)

        cv_results[phase] = {
            'phase':          phase,
            'start_frame':    start_frame,
            'end_frame':      end_frame,
            'window_length':  wl,
            'time':           cfg['time'],
            'n_samples':      win['n_samples'],
            'total_cells':    win['total_cells'],
            'normal_cells':   win['normal_cells'],
            'ips_cells':      win['ips_cells'],
            'balance_ratio':  win['balance_ratio'],
            'survival_methodology_used': win.get('survival_methodology_used', False),
            'baseline_cell_count': win.get('baseline_cell_count', 0),
            'n_evaluations':  agg.get('n_evaluations', 0),
            'metrics':        agg,
            'fold_metrics':   fold_list,
        }

        print(f"\n  Cross-validation complete ({agg.get('n_evaluations',0)} evaluations):")
        print(f"     Precision     : {agg.get('precision_mean',0):.3f} +/- {agg.get('precision_std',0):.3f}")
        print(f"     Recall        : {agg.get('recall_mean',0):.3f} +/- {agg.get('recall_std',0):.3f}")
        print(f"     F1            : {agg.get('f1_mean',0):.3f} +/- {agg.get('f1_std',0):.3f}")
        print(f"     Balanced Acc  : {agg.get('balanced_accuracy_mean',0):.3f} +/- {agg.get('balanced_accuracy_std',0):.3f}")
        print(f"     MCC           : {agg.get('mcc_mean',0):.3f} +/- {agg.get('mcc_std',0):.3f}")

    return cv_results

In [ ]:
# =============================================================================
# CELL 9: HOLDOUT VALIDATION EXPERIMENT
# =============================================================================

def perform_holdout_validation(model_name, n_repeats=N_REPEATS):
    print("\n" + "="*80)
    print(f"EXPERIMENT 2: HOLDOUT VALIDATION - {model_name}")
    print("="*80)
    print("Training on optimal window (I), testing on I-3 through I+3")
    print("Offset 0 : 80/20 stratified holdout split (train 80%, test 20% held-out)")
    print("Others   : full train, fresh window test (independent balanced dataset)")
    print(f"Repeats  : {n_repeats}")
    print("="*80)

    holdout_results = {}

    for phase in [1, 2, 3, 4]:
        cfg         = HOLDOUT_CONFIGS[phase]
        train_start = cfg['train_start']
        train_end   = train_start + WINDOW_LENGTH - 1

        print(f"\nPhase {phase}: Training on Frame {train_start} "
              f"(window {train_start}–{train_end})")
        print("-"*60)

        train_win = feature_processor.create_time_window_data(
            train_start, WINDOW_LENGTH)
        if train_win is None:
            print(f"  ✗ Failed to create training data for Phase {phase}")
            continue

        feature_processor.print_dataset_summary(train_win, "Training dataset")
        X_train, y_train = train_win['X'], train_win['y']

        phase_res = {
            'phase':       phase,
            'train_start': train_start,
            'train_end':   train_end,
            'train_window': cfg['train_window'],
            'train_normal': train_win['normal_cells'],
            'train_ips':   train_win['ips_cells'],
            'train_samples': train_win['n_samples'],
            'neighbor_results': {},
        }

        for test_start, offset, test_window in zip(
                cfg['test_starts'], cfg['offsets'], cfg['test_windows']):

            test_end = test_start + WINDOW_LENGTH - 1
            tag      = f"offset {offset:+d} (window {test_window})"
            print(f"\n  Testing Frame {test_start}, {tag} … ", end="", flush=True)

            if offset == 0:
                X_tr80, X_te20, y_tr80, y_te20 = train_test_split(
                    X_train, y_train,
                    test_size=0.20,
                    stratify=y_train,
                    random_state=42)
                agg, fold_list = evaluation_engine.evaluate_on_test_set(
                    X_tr80, y_tr80, X_te20, y_te20, model_name, n_repeats=n_repeats)
                n_test = len(X_te20)
            else:
                test_win = feature_processor.create_time_window_data(
                    test_start, WINDOW_LENGTH)
                if test_win is None:
                    print("No data")
                    continue
                X_test, y_test = test_win['X'], test_win['y']
                agg, fold_list = evaluation_engine.evaluate_on_test_set(
                    X_train, y_train, X_test, y_test, model_name, n_repeats=n_repeats)
                n_test = len(X_test)

            pr_m = agg.get('precision_mean',         0.0)
            pr_s = agg.get('precision_std',          0.0)
            rc_m = agg.get('recall_mean',            0.0)
            rc_s = agg.get('recall_std',             0.0)
            f1_m = agg.get('f1_mean',                0.0)
            f1_s = agg.get('f1_std',                 0.0)
            ba_m = agg.get('balanced_accuracy_mean', 0.0)
            ba_s = agg.get('balanced_accuracy_std',  0.0)
            mc_m = agg.get('mcc_mean',               0.0)
            mc_s = agg.get('mcc_std',                0.0)
            print(f"Precision: {pr_m:.3f} +/- {pr_s:.3f},  "
                  f"Recall: {rc_m:.3f} +/- {rc_s:.3f},  "
                  f"F1: {f1_m:.3f} +/- {f1_s:.3f},  "
                  f"BalAcc: {ba_m:.3f} +/- {ba_s:.3f},  "
                  f"MCC: {mc_m:.3f} +/- {mc_s:.3f}")

            phase_res['neighbor_results'][offset] = {
                'test_start':  test_start,
                'test_end':    test_end,
                'test_window': test_window,
                'offset':      offset,
                'n_samples':   n_test,
                'metrics':     agg,
                'fold_metrics': fold_list,
            }

        holdout_results[phase] = phase_res

    return holdout_results

In [ ]:
# =============================================================================
# CELL 10: INDEPENDENT TEST EXPERIMENT
# =============================================================================

def perform_independent_test(model_name, n_repeats=N_REPEATS):
    print("\n" + "="*80)
    print(f"EXPERIMENT 3: INDEPENDENT TEST - {model_name}")
    print("="*80)
    print("Train on phase P optimal window → test on distant windows from next phase")
    for ph in [1, 2, 3, 4]:
        cfg = INDEPENDENT_CONFIGS[ph]
        ts  = cfg['test_starts']
        print(f"  Phase {ph} ({cfg['train_window']}) "
              f"→ Phase {cfg['test_phase']}: {','.join(str(s) for s in ts)}")
    print(f"  Window length     : {WINDOW_LENGTH} frames")
    print("="*80)

    independent_results = {}

    for train_phase in [1, 2, 3, 4]:
        cfg         = INDEPENDENT_CONFIGS[train_phase]
        train_start = cfg['train_start']
        train_end   = train_start + WINDOW_LENGTH - 1
        test_phase  = cfg['test_phase']

        print(f"\nTraining Phase {train_phase}: Frame {train_start} "
              f"(window {train_start}–{train_end}, {WINDOW_LENGTH} frames)")
        print(f"Testing on Phase {test_phase} windows (step 3 frames)")
        print("="*60)

        # Load training phase data ONLY (Phase P) - NO test phase data in training
        train_win = feature_processor.create_time_window_data(
            train_start, WINDOW_LENGTH)
        if train_win is None:
            print(f"  Failed to create training data for Phase {train_phase}")
            continue

        feature_processor.print_dataset_summary(train_win, "Training dataset (Phase P only)")
        
        # Get raw training data for normalization fitting
        X_train_raw = train_win['X_raw']
        y_train = train_win['y']
        
        # Normalize using training data only (fit scaler on Phase P)
        lo = np.percentile(X_train_raw, 1, axis=0)
        hi = np.percentile(X_train_raw, 99, axis=0)
        med = np.median(X_train_raw, axis=0)
        q25 = np.percentile(X_train_raw, 25, axis=0)
        q75 = np.percentile(X_train_raw, 75, axis=0)
        iqr = q75 - q25
        iqr[iqr == 0] = 1.0
        
        X_train_clipped = np.clip(X_train_raw, lo, hi)
        X_train_norm = (X_train_clipped - med) / iqr
        
        phase_res = {
            'train_phase':   train_phase,
            'train_start':   train_start,
            'train_end':     train_end,
            'train_window':  cfg['train_window'],
            'train_normal':  train_win['normal_cells'],
            'train_ips':     train_win['ips_cells'],
            'train_samples': train_win['n_samples'],
            'test_phase':    test_phase,
            'test_results':  {},
        }

        # 5-fold CV on training data only (Phase P)
        from sklearn.model_selection import StratifiedKFold
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        cv_models = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_norm, y_train)):
            X_fold_train = X_train_norm[tr_idx]
            y_fold_train = y_train[tr_idx]
            
            model = model_factory.create_model(model_name, random_state=42 + fold)
            if model is None:
                print(f"  Warning: {model_name} not available, skipping fold {fold}")
                continue
            model.fit(X_fold_train, y_fold_train)
            cv_models.append(model)

        if not cv_models:
            print(f"  No models available for {model_name}, skipping Phase {train_phase}")
            continue

        # Store test data for each window (NO sampling - use full test windows)
        test_windows_data = []
        
        for test_start, test_window, label in zip(
                cfg['test_starts'], cfg['test_windows'], cfg['test_labels']):
            
            test_win = feature_processor.create_time_window_data(
                test_start, WINDOW_LENGTH)
            
            if test_win is None:
                print(f"  Warning: Could not create test window {test_window}")
                continue
            
            X_test_raw = test_win['X_raw']
            y_test = test_win['y']
            
            # Apply same normalization (fitted on Phase P training data)
            X_test_clipped = np.clip(X_test_raw, lo, hi)
            X_test_norm = (X_test_clipped - med) / iqr
            
            test_windows_data.append({
                'label': label,
                'test_start': test_start,
                'test_window': test_window,
                'test_end': test_start + WINDOW_LENGTH - 1,
                'X_test': X_test_norm,
                'y_test': y_test,
                'n_test_samples': len(y_test),
                'normal_cells': test_win['normal_cells'],
                'ips_cells': test_win['ips_cells'],
            })
        
        if not test_windows_data:
            print(f"  No test windows available for Phase {train_phase}")
            continue

        # Test on each window (using models trained ONLY on Phase P)
        for tw in test_windows_data:
            print(f"\n  {tw['label']}: Frame {tw['test_start']} (window {tw['test_window']}, "
                  f"{WINDOW_LENGTH} frames) … ", end="", flush=True)
            
            X_test = tw['X_test']
            y_test = tw['y_test']
            
            fold_metrics = []
            for model in cv_models:
                y_prob = model.predict_proba(X_test)[:, 1]
                threshold = evaluation_engine.find_optimal_threshold(y_test, y_prob)
                y_pred = (y_prob >= threshold).astype(int)
                fm = evaluation_engine.calculate_comprehensive_metrics(y_test, y_pred, y_prob)
                fold_metrics.append(fm)
            
            agg = evaluation_engine.aggregate_fold_metrics(fold_metrics)

            pr_m = agg.get('precision_mean', 0.0)
            pr_s = agg.get('precision_std', 0.0)
            rc_m = agg.get('recall_mean', 0.0)
            rc_s = agg.get('recall_std', 0.0)
            f1_m = agg.get('f1_mean', 0.0)
            f1_s = agg.get('f1_std', 0.0)
            ba_m = agg.get('balanced_accuracy_mean', 0.0)
            ba_s = agg.get('balanced_accuracy_std', 0.0)
            mc_m = agg.get('mcc_mean', 0.0)
            mc_s = agg.get('mcc_std', 0.0)
            
            print(f"n={tw['n_test_samples']} (N:{tw['normal_cells']}, iPS:{tw['ips_cells']}) → "
                  f"Prec: {pr_m:.3f}+-{pr_s:.3f}, "
                  f"Rec: {rc_m:.3f}+-{rc_s:.3f}, "
                  f"F1: {f1_m:.3f}+-{f1_s:.3f}, "
                  f"BalAcc: {ba_m:.3f}+-{ba_s:.3f}, "
                  f"MCC: {mc_m:.3f}+-{mc_s:.3f}")

            phase_res['test_results'][tw['label']] = {
                'test_start':     tw['test_start'],
                'test_end':       tw['test_end'],
                'test_window':    tw['test_window'],
                'label':          tw['label'],
                'n_test_samples': tw['n_test_samples'],
                'normal_cells':   tw['normal_cells'],
                'ips_cells':      tw['ips_cells'],
                'metrics':        agg,
            }

        independent_results[train_phase] = phase_res

    return independent_results

In [ ]:
import matplotlib.colors as mcolors

# =============================================================================
# SHARED HELPERS
# =============================================================================

def _set_ax_style(ax):
    ax.set_facecolor('#F8F9FB')
    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_color('#222222')
    ax.spines['left'].set_linewidth(1.4)
    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_color('#222222')
    ax.spines['bottom'].set_linewidth(1.4)
    ax.spines['top'].set_visible(True)
    ax.spines['top'].set_color('#aaaaaa')
    ax.spines['top'].set_linewidth(0.6)
    ax.spines['right'].set_visible(True)
    ax.spines['right'].set_color('#aaaaaa')
    ax.spines['right'].set_linewidth(0.6)


def _base_rcparams(font_family):
    return {
        'font.family':       font_family,
        'font.size':         11,
        'axes.titlesize':    12,
        'axes.labelsize':    11,
        'xtick.labelsize':   10,
        'ytick.labelsize':   10,
        'legend.fontsize':   11,
        'figure.dpi':        300,
        'savefig.dpi':       300,
        'axes.linewidth':    1.4,
        'axes.edgecolor':    '#222222',
        'axes.facecolor':    '#F8F9FB',
        'figure.facecolor':  'white',
        'axes.grid':         True,
        'grid.alpha':        0.18,
        'grid.linestyle':    ':',
        'grid.color':        '#888888',
        'xtick.major.width': 1.2,
        'ytick.major.width': 1.2,
    }


def _gradient_bar(ax, x, height, width, color_dark, color_light, bottom=0):
    """Draw one bar filled with a horizontal gradient (dark left -> light right)."""
    if height <= 0:
        return
    cmap = mcolors.LinearSegmentedColormap.from_list(
        '_gb', [color_dark, color_light])
    grad = np.linspace(0, 1, 256).reshape(1, -1)
    ax.imshow(grad, aspect='auto', cmap=cmap,
              extent=[x - width / 2, x + width / 2,
                      bottom, bottom + height],
              zorder=3, origin='lower')


# =============================================================================
# SHARED PALETTE
# gradient: dark (left) -> light (right) for each metric
# =============================================================================
METRIC_GRADIENTS = {
    'precision':         ('#4a0000', '#ffaaaa'),
    'recall':            ('#00004a', '#aaccff'),
    'f1':                ('#002200', '#aaffaa'),
    'balanced_accuracy': ('#1a0030', '#d4aaff'),
    'mcc':               ('#3a1500', '#ffddaa'),
    'accuracy':          ('#001a1a', '#aaffee'),
}
# Mid-tone of each gradient (for legend patches and line elements)
METRIC_MIDTONE = {
    'precision':         '#cc2200',
    'recall':            '#1a5fcc',
    'f1':                '#2a7a2a',
    'balanced_accuracy': '#7b2d8b',
    'mcc':               '#cc5500',
    'accuracy':          '#1a7a7a',
}
METRIC_LABELS_SHARED = {
    'precision':         'Precision',
    'recall':            'Recall',
    'f1':                'F1-Score',
    'balanced_accuracy': 'Balanced Acc',
    'mcc':               'MCC',
    'accuracy':          'Accuracy',
}
# Phase gradients for combined/radar: each phase its own hue
PHASE_GRADIENTS = {
    1: ('#4a0000', '#ffaaaa'),
    2: ('#00004a', '#aaccff'),
    3: ('#002200', '#aaffaa'),
    4: ('#1a0030', '#d4aaff'),
}
PHASE_MIDTONE = {
    1: '#cc2200',
    2: '#1a5fcc',
    3: '#2a7a2a',
    4: '#7b2d8b',
}


In [ ]:
# =============================================================================
# CELL 11: CROSS-VALIDATION FIGURE
# =============================================================================

def plot_cross_validation(cv_results, timestamp):
    if not cv_results:
        print("No CV results to plot.")
        return None

    font_family = setup_publication_fonts()
    plt.rcParams.update(_base_rcparams(font_family))

    phases          = sorted(cv_results.keys())
    x_pos           = np.arange(len(phases))
    bar_w           = 0.6
    x_labels        = [f"Phase {p}" for p in phases]
    # REMOVED 'accuracy' from metrics
    primary_metrics = ['precision', 'recall', 'f1', 'balanced_accuracy', 'mcc']

    # Change to 2 rows, 3 columns (5 metrics, 1 empty)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for ax_idx, metric in enumerate(primary_metrics):
        ax    = axes[ax_idx]
        means = [cv_results[p]['metrics'].get(f'{metric}_mean', 0) for p in phases]
        stds  = [cv_results[p]['metrics'].get(f'{metric}_std',  0) for p in phases]

        c_dark, c_light = METRIC_GRADIENTS[metric]
        for xi, mv in zip(x_pos, means):
            _gradient_bar(ax, xi, mv, bar_w, c_dark, c_light)

        ax.errorbar(x_pos, means, yerr=stds,
                    fmt='none', ecolor='#111111',
                    elinewidth=1.1, capsize=3, capthick=1.1, zorder=5)

        ax.set_xlim(-0.6, len(phases) - 0.4)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(x_labels, fontsize=11, fontweight='bold')
        ax.set_ylabel(METRIC_LABELS_SHARED.get(metric, metric),
                      fontweight='bold', fontsize=12)
        ax.set_ylim(0, 1.08)
        ax.set_yticks(np.arange(0, 1.1, 0.2))
        ax.set_title(METRIC_LABELS_SHARED.get(metric, metric),
                     fontweight='bold', fontsize=13, pad=10)
        _set_ax_style(ax)

    # Hide the 6th subplot (index 5) since we only have 5 metrics
    axes[5].axis('off')

    legend_elements = [
        plt.Rectangle((0, 0), 1, 1,
                       facecolor=METRIC_MIDTONE[m], edgecolor='white',
                       label=METRIC_LABELS_SHARED[m])
        for m in primary_metrics
    ]
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=len(primary_metrics), fontsize=12, frameon=True,
               framealpha=0.97, bbox_to_anchor=(0.5, 0.02),
               edgecolor='#CCCCCC', columnspacing=2.0)

    fig.suptitle(f'Cross-Validation Results: 5-fold x {N_REPEATS} Repeat',
                 fontsize=16, fontweight='bold', y=0.98)

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])

    fname = os.path.join(PLOTS_DIR, f'cross_validation_{timestamp}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Cross-validation figure saved: {fname}")
    plt.show()
    plt.close(fig)
    return fname

In [ ]:
# =============================================================================
# CELL 12: HOLDOUT VALIDATION FIGURE
# =============================================================================

def plot_holdout_validation(holdout_results, timestamp):
    if not holdout_results:
        print("No holdout results to plot.")
        return None

    font_family = setup_publication_fonts()
    rc = _base_rcparams(font_family)
    rc['legend.fontsize'] = 10
    plt.rcParams.update(rc)

    offsets     = [-3, -2, -1, 0, 1, 2, 3]
    key_metrics = ['precision', 'recall', 'f1', 'mcc']  # REMOVED 'balanced_accuracy'
    n_metrics   = len(key_metrics)
    bar_width   = 0.15
    spacing     = 0.02

    fig, axes = plt.subplots(4, 1, figsize=(16, 20))
    fig.subplots_adjust(hspace=0.4, bottom=0.05, top=0.93, left=0.1, right=0.95)

    for p_idx, phase in enumerate([1, 2, 3, 4]):
        ax = axes[p_idx]

        if phase not in holdout_results:
            ax.text(0.5, 0.5, f'No data for Phase {phase}',
                    ha='center', va='center', transform=ax.transAxes)
            _set_ax_style(ax)
            continue

        phase_data       = holdout_results[phase]
        train_start      = phase_data['train_start']
        train_end        = phase_data['train_end']
        neighbor_results = phase_data['neighbor_results']
        x_positions      = np.arange(len(offsets))

        for m_idx, metric in enumerate(key_metrics):
            offset_pos    = (m_idx - n_metrics / 2 + 0.5) * (bar_width + spacing)
            bar_positions = x_positions + offset_pos
            c_dark, c_light = METRIC_GRADIENTS[metric]

            means = [neighbor_results[o]['metrics'].get(f'{metric}_mean', 0)
                     if o in neighbor_results else 0 for o in offsets]
            stds  = [neighbor_results[o]['metrics'].get(f'{metric}_std',  0)
                     if o in neighbor_results else 0 for o in offsets]

            for bx, mv in zip(bar_positions, means):
                _gradient_bar(ax, bx, mv, bar_width, c_dark, c_light)

            ax.errorbar(bar_positions, means, yerr=stds,
                        fmt='none', ecolor='#111111',
                        elinewidth=1.1, capsize=3, capthick=1.1, zorder=5,
                        label=METRIC_LABELS_SHARED[metric] if p_idx == 0 else "")

        ax.axvspan(3 - 0.4, 3 + 0.4, alpha=0.08, color='#FFD700', zorder=0)
        ax.set_xlim(-0.6, len(offsets) - 0.4)
        ax.set_xticks(x_positions)
        ax.set_xticklabels([f'{o:+d}' for o in offsets], fontsize=11)
        ax.set_xlabel('Offset from training window', fontweight='bold', fontsize=12)
        ax.set_ylabel('Score', fontweight='bold', fontsize=12)
        ax.set_ylim(0, 1.08)
        ax.set_yticks(np.arange(0, 1.1, 0.2))

        train_time = frame_to_time(train_start)
        ax.set_title(
            f'Phase {phase}: Train at frames {train_start}-{train_end} ({train_time})',
            fontweight='bold', fontsize=13, pad=15)
        _set_ax_style(ax)

    legend_elements = [
        plt.Rectangle((0, 0), 1, 1,
                       facecolor=METRIC_MIDTONE[m], edgecolor='white',
                       label=METRIC_LABELS_SHARED[m])
        for m in key_metrics
    ]
    legend_elements.append(
        plt.Rectangle((0, 0), 1, 1, facecolor='#FFD700', alpha=0.3,
                       edgecolor='none', label='Offset 0 (80/20 holdout)')
    )

    fig.legend(handles=legend_elements, loc='upper center',
               ncol=len(legend_elements), fontsize=11, frameon=True,
               framealpha=0.97, bbox_to_anchor=(0.5, 0.97),
               edgecolor='#CCCCCC', columnspacing=1.5)

    plt.tight_layout(rect=[0, 0.02, 1, 0.95])

    fname = os.path.join(PLOTS_DIR, f'holdout_validation_{timestamp}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Holdout figure saved: {fname}")
    plt.show()
    plt.close(fig)
    return fname

In [ ]:
# =============================================================================
# CELL 13: INDEPENDENT TEST FIGURE
# =============================================================================

def plot_independent_test(independent_results, timestamp):
    if not independent_results:
        print("No independent test results to plot.")
        return None

    font_family = setup_publication_fonts()
    rc = _base_rcparams(font_family)
    rc['xtick.labelsize'] = 9
    rc['legend.fontsize'] = 10
    plt.rcParams.update(rc)

    key_metrics  = ['precision', 'recall', 'f1', 'balanced_accuracy', 'mcc']
    n_metrics    = len(key_metrics)
    bar_width    = 0.15
    spacing      = 0.02
    # FIXED: Correct training windows for each phase
    phase_titles = [
        'Phase 1: Train at Frame 12-40',
        'Phase 2: Train at Frame 108-136',
        'Phase 3: Train at Frame 152-180',
        'Phase 4: Train at Frame 233-261',
    ]

    fig, axes = plt.subplots(4, 1, figsize=(16, 20))
    fig.subplots_adjust(hspace=0.4, bottom=0.1, top=0.95, left=0.1, right=0.95)

    for p_idx, train_phase in enumerate([1, 2, 3, 4]):
        ax = axes[p_idx]

        if train_phase not in independent_results:
            ax.text(0.5, 0.5, f'No data for Phase {train_phase}',
                    ha='center', va='center', transform=ax.transAxes)
            _set_ax_style(ax)
            continue

        phase_results = independent_results[train_phase]
        train_start   = phase_results['train_start']
        train_end     = phase_results['train_end']
        test_results  = phase_results['test_results']

        sorted_items = sorted(test_results.items(),
                              key=lambda x: x[1]['test_start'])
        test_data    = [item[1] for item in sorted_items]
        test_starts  = [data['test_start'] for data in test_data]
        n_tests      = len(test_starts)
        x_positions  = np.arange(n_tests)
        x_labels     = [f"{s}/{frame_to_time(s)}" for s in test_starts]

        for m_idx, metric in enumerate(key_metrics):
            offset_pos    = (m_idx - n_metrics / 2 + 0.5) * (bar_width + spacing)
            bar_positions = x_positions + offset_pos
            c_dark, c_light = METRIC_GRADIENTS[metric]

            means = [data['metrics'].get(f'{metric}_mean', 0) for data in test_data]
            stds  = [data['metrics'].get(f'{metric}_std',  0) for data in test_data]

            for bx, mv in zip(bar_positions, means):
                _gradient_bar(ax, bx, mv, bar_width, c_dark, c_light)

            ax.errorbar(bar_positions, means, yerr=stds,
                        fmt='none', ecolor='#111111',
                        elinewidth=1.1, capsize=3, capthick=1.1, zorder=5,
                        label=METRIC_LABELS_SHARED[metric] if p_idx == 0 else "")

        ax.set_xlim(-0.55, n_tests - 0.45)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels, rotation=38, ha='right',
                           fontsize=9, fontweight='bold')
        ax.set_ylabel('Score', fontweight='bold', fontsize=12)
        ax.set_ylim(0, 1.08)
        ax.set_yticks(np.arange(0, 1.1, 0.2))

        # FIXED: Use phase_titles instead of reconstructing
        ax.set_title(phase_titles[p_idx], fontweight='bold', fontsize=13, pad=15)
        _set_ax_style(ax)

    legend_elements = [
        plt.Rectangle((0, 0), 1, 1,
                       facecolor=METRIC_MIDTONE[m], edgecolor='white',
                       label=METRIC_LABELS_SHARED[m])
        for m in key_metrics
    ]
    fig.legend(handles=legend_elements, loc='lower center',
               ncol=n_metrics, fontsize=11, frameon=True,
               framealpha=0.97, bbox_to_anchor=(0.5, 0.02),
               edgecolor='#CCCCCC', columnspacing=1.5)

    fig.suptitle('Independent Test: Cross-Phase Generalization',
                 fontsize=16, fontweight='bold', y=0.98)

    plt.tight_layout(rect=[0, 0.05, 1, 0.96])

    fname = os.path.join(PLOTS_DIR, f'independent_test_{timestamp}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Independent test figure saved: {fname}")
    plt.show()
    plt.close(fig)
    return fname

In [ ]:
# =============================================================================
# CELL 14: COMBINED SUMMARY FIGURE
# =============================================================================

def plot_combined_summary(cv_results, holdout_results, independent_results,
                          timestamp, model_name="GB-LGBM Hybrid"):

    font_family = setup_publication_fonts()
    rc = _base_rcparams(font_family)
    rc.update({'axes.titlesize': 11, 'axes.labelsize': 10,
               'xtick.labelsize': 9, 'ytick.labelsize': 9,
               'legend.fontsize': 9})
    plt.rcParams.update(rc)

    fig = plt.figure(figsize=(26, 20))
    gs  = gridspec.GridSpec(3, 4, figure=fig,
                            hspace=0.40, wspace=0.30,
                            left=0.06, right=0.98,
                            top=0.92, bottom=0.10)

    summary_metrics = ['precision', 'f1', 'mcc']
    row_titles      = ['Cross-Validation', 'Holdout Validation', 'Independent Test']
    phases          = sorted(cv_results.keys())
    x_pos           = np.arange(len(phases))

    # Row 0 — cross-validation: one metric per subplot, gradient bars per phase
    for col_i, metric in enumerate(summary_metrics):
        ax    = fig.add_subplot(gs[0, col_i])
        means = [cv_results[p]['metrics'].get(f'{metric}_mean', 0) for p in phases]
        stds  = [cv_results[p]['metrics'].get(f'{metric}_std',  0) for p in phases]

        c_dark, c_light = METRIC_GRADIENTS[metric]
        for xi, mv in zip(x_pos, means):
            _gradient_bar(ax, xi, mv, 0.55, c_dark, c_light)

        ax.errorbar(x_pos, means, yerr=stds,
                    fmt='none', ecolor='#111111',
                    elinewidth=1.1, capsize=3, capthick=1.1, zorder=5)

        for xi, mv, sv in zip(x_pos, means, stds):
            if mv + sv < 0.98:
                ax.text(xi, mv + sv + 0.012, f'{mv:.3f}',
                        ha='center', va='bottom',
                        fontsize=9, fontweight='bold')

        ax.set_xlim(-0.5, len(phases) - 0.5)
        ax.set_xticks(x_pos)
        ax.set_xticklabels([f'Ph{p}' for p in phases], fontsize=10)
        ax.set_ylim(0, 1.08)
        ax.yaxis.set_major_locator(MultipleLocator(0.2))
        ax.axhline(0.6, color=METRIC_MIDTONE[metric], linestyle='--',
                   linewidth=1.2, alpha=0.55)
        ax.set_title(METRIC_LABELS_SHARED.get(metric, metric),
                     fontweight='bold', fontsize=11)
        ax.set_ylabel(METRIC_LABELS_SHARED.get(metric, metric), fontsize=10)
        _set_ax_style(ax)

    ax_leg = fig.add_subplot(gs[0, 3])
    ax_leg.axis('off')
    handles = [mpatches.Patch(color=METRIC_MIDTONE[m],
                               label=METRIC_LABELS_SHARED.get(m, m))
               for m in summary_metrics]
    handles.append(plt.Line2D([0], [0], color='#444444', linestyle='--',
                               linewidth=1.5, label='60% threshold'))
    ax_leg.legend(handles=handles, loc='center', fontsize=11,
                  frameon=True, framealpha=0.97)
    ax_leg.set_title('Cross-Validation', fontweight='bold', fontsize=12)

    # Row 1 — holdout: precision gradient bars, gold for offset 0
    offsets   = [-3, -2, -1, 0, 1, 2, 3]
    ho_metric = 'precision'
    c_dark_p, c_light_p = METRIC_GRADIENTS['precision']

    for p_idx, phase in enumerate([1, 2, 3, 4]):
        ax = fig.add_subplot(gs[1, p_idx])
        if phase not in holdout_results:
            ax.axis('off')
            continue

        nr    = holdout_results[phase]['neighbor_results']
        means = [nr[o]['metrics'].get(f'{ho_metric}_mean', 0)
                 if o in nr else 0 for o in offsets]
        stds  = [nr[o]['metrics'].get(f'{ho_metric}_std',  0)
                 if o in nr else 0 for o in offsets]
        x     = np.arange(len(offsets))

        for xi, o, mv in zip(x, offsets, means):
            if o == 0:
                _gradient_bar(ax, xi, mv, 0.7, '#7a5a00', '#ffe680')
            else:
                _gradient_bar(ax, xi, mv, 0.7, c_dark_p, c_light_p)

        ax.errorbar(x, means, yerr=stds,
                    fmt='none', ecolor='#111111',
                    elinewidth=1.1, capsize=3, capthick=1.1, zorder=5)

        for xi, mv in zip(x, means):
            if 0 < mv < 0.95:
                ax.text(xi, mv + 0.018, f'{mv:.2f}',
                        ha='center', va='bottom',
                        fontsize=8, fontweight='bold')

        ax.set_xlim(-0.5, len(offsets) - 0.5)
        ax.set_xticks(x)
        ax.set_xticklabels([f'{o:+d}' for o in offsets], fontsize=8)
        ax.set_ylim(0, 1.08)
        ax.yaxis.set_major_locator(MultipleLocator(0.2))
        ax.axhline(0.6, color=METRIC_MIDTONE['precision'], linestyle='--',
                   linewidth=1.2, alpha=0.55)
        ax.set_title(f'Phase {phase}\n(Precision, offset 0 = 80/20)',
                     fontweight='bold', fontsize=10, pad=8)
        ax.set_xlabel('Offset', fontsize=9)
        _set_ax_style(ax)

    # Row 2 — independent test: precision gradient bars per phase
    it_metric = 'precision'
    it_titles = ['Ph1 (temporal)', 'Ph2 (temporal)',
                 'Ph3 (temporal)', 'Ph4 (temporal)']

    for p_idx, train_phase in enumerate([1, 2, 3, 4]):
        ax = fig.add_subplot(gs[2, p_idx])
        if train_phase not in independent_results:
            ax.axis('off')
            continue

        tr     = independent_results[train_phase]['test_results']
        labels = list(tr.keys())
        if not labels:
            ax.axis('off')
            continue

        means = [tr[lb]['metrics'].get(f'{it_metric}_mean', 0) for lb in labels]
        stds  = [tr[lb]['metrics'].get(f'{it_metric}_std',  0) for lb in labels]
        x     = np.arange(len(labels))

        for xi, mv in zip(x, means):
            _gradient_bar(ax, xi, mv, 0.7, c_dark_p, c_light_p)

        ax.errorbar(x, means, yerr=stds,
                    fmt='none', ecolor='#111111',
                    elinewidth=1.1, capsize=3, capthick=1.1, zorder=5)

        for xi, mv in zip(x, means):
            if 0 < mv < 0.95:
                ax.text(xi, mv + 0.018, f'{mv:.2f}',
                        ha='center', va='bottom',
                        fontsize=8, fontweight='bold')

        ax.set_xlim(-0.5, len(labels) - 0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylim(0, 1.08)
        ax.yaxis.set_major_locator(MultipleLocator(0.2))
        ax.axhline(0.6, color=METRIC_MIDTONE['precision'], linestyle='--',
                   linewidth=1.2, alpha=0.55)
        ax.set_title(f'{it_titles[p_idx]}\n(Precision)',
                     fontweight='bold', fontsize=10, pad=8)
        ax.set_xlabel('Test window', fontsize=9)
        _set_ax_style(ax)

    for i, title in enumerate(row_titles):
        fig.text(0.02, 0.80 - i * 0.28, title,
                 fontsize=14, fontweight='bold',
                 rotation=90, va='center', ha='center')

    fig.suptitle(
        f'Complete Validation Summary - {model_name}\n'
        f'5-fold CV x {N_REPEATS} repeat, Balanced 10:1, Window = {WINDOW_LENGTH} frames',
        fontsize=16, fontweight='bold', y=0.97)

    fname = os.path.join(PLOTS_DIR, f'combined_summary_{model_name.replace(" ", "_")}_{timestamp}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Combined summary figure saved: {fname}")
    plt.show()
    plt.close(fig)
    return fname

In [ ]:
# =============================================================================
# CELL 15: RADAR CHART
# =============================================================================

def plot_radar_cv(cv_results, timestamp):
    if not cv_results:
        return None

    font_family = setup_publication_fonts()
    plt.rcParams.update(_base_rcparams(font_family))

    radar_metrics = ['precision', 'recall', 'f1', 'balanced_accuracy', 'mcc']
    angles        = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
    angles       += angles[:1]

    fig, ax = plt.subplots(1, 1, figsize=(10, 10),
                           subplot_kw={'projection': 'polar'})

    ax.set_facecolor('#F8F9FB')
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_rlim(0, 1.0)
    ax.set_rticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=10)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(
        [METRIC_LABELS_SHARED.get(m, m) for m in radar_metrics],
        fontsize=12, fontweight='bold')
    ax.yaxis.grid(True, color='#888888', alpha=0.18, linewidth=0.8, linestyle=':')
    ax.xaxis.grid(True, color='#888888', alpha=0.18, linewidth=0.8, linestyle=':')

    for phase in sorted(cv_results.keys()):
        vals  = [cv_results[phase]['metrics'].get(f'{m}_mean', 0)
                 for m in radar_metrics]
        vals += vals[:1]
        clr   = PHASE_MIDTONE[phase]
        ax.plot(angles, vals, 'o-', color=clr, linewidth=2.5,
                markersize=8, label=f'Phase {phase}')
        ax.fill(angles, vals, alpha=0.12, color=clr)

    ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.08),
              ncol=4, fontsize=11, frameon=True,
              framealpha=0.97, edgecolor='#CCCCCC', columnspacing=1.5)

    ax.set_title('Cross-Validation Radar Chart',
                 fontsize=16, fontweight='bold', pad=20)

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])

    fname = os.path.join(PLOTS_DIR, f'radar_chart_{timestamp}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Radar chart saved: {fname}")
    plt.show()
    plt.close(fig)
    return fname

In [ ]:
# =============================================================================
# CELL 16: SAVE ALL RESULTS TO JSON + TXT
# =============================================================================

def save_all_results(cv_results, holdout_results, independent_results, ts, model_name="GB-LGBM Hybrid"):
    """Persist all numeric results to JSON and a readable text summary."""

    def _safe(v):
        if isinstance(v, (np.integer, np.int64, np.int32)):
            return int(v)
        if isinstance(v, (np.floating, np.float64, np.float32)):
            return float(v)
        if isinstance(v, np.ndarray):
            return v.tolist()
        return v

    def _clean(d):
        if isinstance(d, dict):
            return {k: _clean(v) for k, v in d.items()
                    if k not in ('X','y','fold_metrics','window_data')}
        if isinstance(d, (list, tuple)):
            return [_clean(i) for i in d]
        return _safe(d)

    bundle = {
        'timestamp':        ts,
        'model_name':       model_name,
        'window_length':    WINDOW_LENGTH,
        'balance_ratio':    10,
        'n_cv_repeats':     N_REPEATS,
        'best_configs':     {str(k): v for k,v in BEST_CONFIGS.items()},
        'cross_validation': _clean({str(k): v for k,v in cv_results.items()}),
        'holdout_validation': _clean({str(k): v for k,v in holdout_results.items()}),
        'independent_test': _clean({str(k): v for k,v in independent_results.items()}),
    }
    json_file = os.path.join(DATA_DIR, f'validation_results_{model_name.replace(" ", "_")}_{ts}.json')
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(bundle, f, indent=2, ensure_ascii=False)
    print(f"JSON results saved : {json_file}")

    txt_file = os.path.join(DATA_DIR, f'validation_summary_{model_name.replace(" ", "_")}_{ts}.txt')
    sep80  = "="*80 + "\n"
    sep60  = "-"*60 + "\n"

    with open(txt_file, 'w', encoding='utf-8') as f:
        f.write(sep80)
        f.write(f"{model_name} VALIDATION SUMMARY\n")
        f.write(sep80)
        f.write(f"Timestamp      : {ts}\n")
        f.write(f"Model          : {model_name}\n")
        f.write(f"Window length  : {WINDOW_LENGTH} frames\n")
        f.write(f"Balance ratio  : 10:1  (Normal:iPS)\n")
        f.write(f"CV repeats     : {N_REPEATS}\n\n")

        f.write(sep80)
        f.write("EXPERIMENT 1: CROSS-VALIDATION\n")
        f.write(sep80)
        metric_order = ['precision','recall','f1','f05','f2',
                        'balanced_accuracy','accuracy','mcc',
                        'roc_auc','pr_auc','sensitivity','specificity']
        for phase in sorted(cv_results.keys()):
            d = cv_results[phase]
            f.write(f"\nPhase {phase}: frames {d['start_frame']}-{d['end_frame']}  "
                    f"({frame_to_time(d['start_frame'])})  "
                    f"N={d['n_samples']} (norm={d['normal_cells']}, iPS={d['ips_cells']})  "
                    f"ratio={d['balance_ratio']:.1f}:1  "
                    f"evals={d['n_evaluations']}\n")
            f.write(sep60)
            for mk in metric_order:
                m_key = f'{mk}_mean'; s_key = f'{mk}_std'
                if m_key in d['metrics']:
                    lbl = METRIC_LABELS.get(mk, mk)
                    f.write(f"   {lbl:22s}: {d['metrics'][m_key]:.4f} +/- "
                            f"{d['metrics'][s_key]:.4f}\n")

        f.write("\n" + sep80)
        f.write("EXPERIMENT 2: HOLDOUT VALIDATION\n")
        f.write(sep80)
        for phase in sorted(holdout_results.keys()):
            d    = holdout_results[phase]
            nr   = d['neighbor_results']
            f.write(f"\nPhase {phase}: Train {d['train_window']}  "
                    f"N_train={d['train_samples']}\n")
            f.write(sep60)
            hdr = (f"   {'Offset':>7}  {'Window':>10}  "
                   + "  ".join(f"{METRIC_LABELS.get(m,m):>10}" for m in
                               ['precision','recall','f1',
                                'balanced_accuracy','mcc']) + "\n")
            f.write(hdr)
            for offset in [-3,-2,-1,0,1,2,3]:
                if offset not in nr:
                    continue
                r  = nr[offset]
                cv_flag = "(80/20)" if offset == 0 else "      "
                line   = (f"   {offset:>+7}{cv_flag:<7}  "
                          f"{r['test_window']:>10}  ")
                for mk in ['precision','recall','f1',
                           'balanced_accuracy','mcc']:
                    mv = r['metrics'].get(f'{mk}_mean', 0)
                    sv = r['metrics'].get(f'{mk}_std',  0)
                    line += f"  {mv:.3f}+-{sv:.3f}"
                f.write(line + "\n")

        f.write("\n" + sep80)
        f.write("EXPERIMENT 3: INDEPENDENT TEST\n")
        f.write(sep80)
        for train_phase in sorted(independent_results.keys()):
            d  = independent_results[train_phase]
            tr = d['test_results']
            f.write(f"\nTrain Phase {train_phase}: {d['train_window']}  "
                    f"N_train={d['train_samples']}\n")
            f.write(sep60)
            hdr = (f"   {'Label':>5}  {'Window':>10}  "
                   + "  ".join(f"{METRIC_LABELS.get(m,m):>10}" for m in
                               ['precision','recall','f1',
                                'balanced_accuracy','mcc']) + "\n")
            f.write(hdr)
            for label, res in tr.items():
                line = (f"   {label:>5}  {res['test_window']:>10}  ")
                for mk in ['precision','recall','f1',
                           'balanced_accuracy','mcc']:
                    mv = res['metrics'].get(f'{mk}_mean', 0)
                    sv = res['metrics'].get(f'{mk}_std',  0)
                    line += f"  {mv:.3f}+-{sv:.3f}"
                f.write(line + "\n")

    print(f"Text summary saved  : {txt_file}")
    return json_file, txt_file

In [ ]:
# =============================================================================
# CELL 17: PRINT COMPREHENSIVE STATISTICS
# =============================================================================

def print_comprehensive_statistics(cv_results, holdout_results,
                                    independent_results):
    """Terminal summary of all three experiments."""
    sep = "="*80

    print("\n" + sep)
    print("COMPREHENSIVE VALIDATION STATISTICS")
    print(sep)

    print("\nEXPERIMENT 1: CROSS-VALIDATION")
    print("-"*60)
    print(f"{'Phase':>7}  {'Window':>10}  {'N':>7}  "
          f"{'Prec':>8}  {'Rec':>8}  {'F1':>8}  "
          f"{'BalAcc':>8}  {'MCC':>8}")
    print("-"*60)
    for phase in sorted(cv_results.keys()):
        d  = cv_results[phase]
        m  = d['metrics']
        print(f"  {phase:>5}  {d['start_frame']:>4}-{d['end_frame']:>4}  "
              f"{d['n_samples']:>7}  "
              f"{m.get('precision_mean',0):>8.4f}  "
              f"{m.get('recall_mean',0):>8.4f}  "
              f"{m.get('f1_mean',0):>8.4f}  "
              f"{m.get('balanced_accuracy_mean',0):>8.4f}  "
              f"{m.get('mcc_mean',0):>8.4f}")

    print("\nEXPERIMENT 2: HOLDOUT VALIDATION")
    print("-"*60)
    for phase in sorted(holdout_results.keys()):
        d  = holdout_results[phase]
        nr = d['neighbor_results']
        print(f"\n  Phase {phase}  (Train: {d['train_window']}):")
        print(f"  {'Offset':>8}  {'Window':>10}  {'N':>7}  "
              f"{'Prec':>8}  {'Rec':>8}  {'F1':>8}  "
              f"{'BalAcc':>8}  {'MCC':>8}")
        print("  " + "-"*67)
        for offset in [-3,-2,-1,0,1,2,3]:
            if offset not in nr:
                continue
            r  = nr[offset]
            m  = r['metrics']
            cv_flag = "(80/20)" if offset == 0 else "      "
            print(f"  {offset:>+8}{cv_flag}  "
                  f"{r['test_window']:>10}  "
                  f"{r['n_samples']:>7}  "
                  f"{m.get('precision_mean',0):>8.4f}  "
                  f"{m.get('recall_mean',0):>8.4f}  "
                  f"{m.get('f1_mean',0):>8.4f}  "
                  f"{m.get('balanced_accuracy_mean',0):>8.4f}  "
                  f"{m.get('mcc_mean',0):>8.4f}")

    print("\nEXPERIMENT 3: INDEPENDENT TEST")
    print("-"*60)
    for train_phase in sorted(independent_results.keys()):
        d  = independent_results[train_phase]
        tr = d['test_results']
        print(f"\n  Train Phase {train_phase} ({d['train_window']}) → "
              f"Test Phase {d['test_phase']}:")
        print(f"  {'Label':>6}  {'Window':>10}  {'N_test':>7}  "
              f"{'Prec':>8}  {'Rec':>8}  {'F1':>8}  "
              f"{'BalAcc':>8}  {'MCC':>8}")
        print("  " + "-"*65)
        for label, res in tr.items():
            m = res['metrics']
            print(f"  {label:>6}  {res['test_window']:>10}  "
                  f"{res.get('n_test_samples',0):>7}  "
                  f"{m.get('precision_mean',0):>8.4f}  "
                  f"{m.get('recall_mean',0):>8.4f}  "
                  f"{m.get('f1_mean',0):>8.4f}  "
                  f"{m.get('balanced_accuracy_mean',0):>8.4f}  "
                  f"{m.get('mcc_mean',0):>8.4f}")

    print("\n" + sep)


In [ ]:
# =============================================================================
# CELL 18: MAIN EXECUTION HEADER
# =============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE VALIDATION — MULTI-MODEL COMPARISON")
print("="*80)
print(f"CV folds: 5 × {N_REPEATS} repeat(s) = {5*N_REPEATS} evaluations per phase")
print(f"Balance : 10:1 (FOV-aware, daughters excluded)")
print(f"Models  : {len(model_factory.get_available_models())}")
print("="*80 + "\n")

# Get available models
available_models = model_factory.get_available_models()
print("Models to evaluate:")
for i, m in enumerate(available_models, 1):
    print(f"  {i}. {m}")
print("="*80 + "\n")

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# Dictionary to store all results for comparison
all_cv_results = {}
all_holdout_results = {}
all_independent_results = {}

print("Ready to run experiments. Proceed to CELL 19.")

In [ ]:
# =============================================================================
# CELL 19: RUN CROSS-VALIDATION (ALL MODELS)
# =============================================================================

print("\n" + "="*80)
print("RUNNING CROSS-VALIDATION EXPERIMENT - ALL MODELS")
print("="*80)

# Get available models
available_models = model_factory.get_available_models()
print(f"\nEvaluating {len(available_models)} models:")
for i, m in enumerate(available_models, 1):
    print(f"  {i}. {m}")
print("="*80)

# Store all CV results
all_cv_results = {}

t0_total = time.time()

for model_name in available_models:
    print("\n" + "="*80)
    print(f"MODEL: {model_name}")
    print("="*80)
    
    t0 = time.time()
    cv_results = perform_cross_validation(model_name, n_repeats=N_REPEATS)
    elapsed = (time.time() - t0) / 60
    print(f"\n  Time: {elapsed:.1f} minutes")
    
    all_cv_results[model_name] = cv_results
    
    # Print summary for this model
    print("\n  RESULTS SUMMARY:")
    print(f"  {'Phase':>7}  {'Window':>10}  {'N':>7}  "
          f"{'Prec':>8}  {'Rec':>8}  {'F1':>8}  {'BalAcc':>8}  {'MCC':>8}")
    print("  " + "-" * 70)
    
    for phase in sorted(cv_results.keys()):
        d = cv_results[phase]
        m = d['metrics']
        print(f"    {phase:>5}  {d['start_frame']:>4}-{d['end_frame']:>4}  "
              f"{d['n_samples']:>7}  "
              f"{m.get('precision_mean', 0):>8.4f}  "
              f"{m.get('recall_mean', 0):>8.4f}  "
              f"{m.get('f1_mean', 0):>8.4f}  "
              f"{m.get('balanced_accuracy_mean', 0):>8.4f}  "
              f"{m.get('mcc_mean', 0):>8.4f}")

print("\n" + "="*80)
print("CROSS-VALIDATION COMPLETE - ALL MODELS")
print(f"Total time: {(time.time()-t0_total)/60:.1f} minutes")
print("="*80)

# Final comparison table across all models
print("\n" + "="*80)
print("CROSS-VALIDATION COMPARISON TABLE (Phase 1 F1 Score)")
print("="*80)
print(f"{'Model':<25}  {'Phase1':>8}  {'Phase2':>8}  {'Phase3':>8}  {'Phase4':>8}")
print("-" * 60)

for model_name, cv_results in all_cv_results.items():
    f1_scores = []
    for phase in [1, 2, 3, 4]:
        if phase in cv_results:
            f1 = cv_results[phase]['metrics'].get('f1_mean', 0)
            f1_scores.append(f"{f1:.4f}")
        else:
            f1_scores.append("N/A")
    print(f"{model_name:<25}  {f1_scores[0]:>8}  {f1_scores[1]:>8}  {f1_scores[2]:>8}  {f1_scores[3]:>8}")

print("="*80)

In [ ]:
# =============================================================================
# CELL 20: PLOT CROSS-VALIDATION RESULTS (ALL MODELS)
# =============================================================================

print("\n" + "="*80)
print("GENERATING CROSS-VALIDATION FIGURES - ALL MODELS")
print("="*80)

# Make sure all_cv_results exists from CELL 19
try:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for model_name, cv_results in all_cv_results.items():
        print(f"\n{'='*60}")
        print(f"Model: {model_name}")
        print(f"{'='*60}")
        
        figs = {}
        
        # Main cross-validation bar chart
        print("→ Creating cross-validation bar chart...")
        figs['cv_main'] = plot_cross_validation(cv_results, f"{timestamp}_{model_name.replace(' ', '_')}")
        
        # Radar chart
        print("→ Creating radar chart...")
        figs['cv_radar'] = plot_radar_cv(cv_results, f"{timestamp}_{model_name.replace(' ', '_')}")
        
        print(f"Figures saved for {model_name}:")
        for name, path in figs.items():
            if path:
                print(f"    • {name}: {path}")
    
    print("\n" + "="*80)
    print("CROSS-VALIDATION FIGURES COMPLETE - ALL MODELS")
    print("="*80)
    
except NameError:
    print("Error: all_cv_results not defined. Run CELL 19 first.")

In [ ]:
# =============================================================================
# CELL 21: RUN HOLDOUT VALIDATION (ALL MODELS)
# =============================================================================

print("\n" + "="*80)
print("RUNNING HOLDOUT VALIDATION EXPERIMENT - ALL MODELS")
print("="*80)

# Get available models
available_models = model_factory.get_available_models()
print(f"\nEvaluating {len(available_models)} models:")
for i, m in enumerate(available_models, 1):
    print(f"  {i}. {m}")
print("="*80)

# Store all holdout results
all_holdout_results = {}

t0_total = time.time()

for model_name in available_models:
    print("\n" + "="*80)
    print(f"MODEL: {model_name}")
    print("="*80)
    
    t0 = time.time()
    holdout_results = perform_holdout_validation(model_name, n_repeats=N_REPEATS)
    elapsed = (time.time() - t0) / 60
    print(f"\n  Time: {elapsed:.1f} minutes")
    
    all_holdout_results[model_name] = holdout_results
    
    # Print summary for this model
    for phase in sorted(holdout_results.keys()):
        d = holdout_results[phase]
        nr = d['neighbor_results']
        print(f"\n  Phase {phase}  (Train: {d['train_window']}):")
        print(f"  {'Offset':>8}  {'Window':>10}  {'N':>7}  "
              f"{'Prec':>8}  {'Rec':>8}  {'F1':>8}  {'BalAcc':>8}  {'MCC':>8}")
        print("  " + "-"*67)
        
        for offset in [-3, -2, -1, 0, 1, 2, 3]:
            if offset not in nr:
                continue
            r = nr[offset]
            m = r['metrics']
            cv_flag = "(80/20)" if offset == 0 else "      "
            print(f"  {offset:>+8}{cv_flag}  "
                  f"{r['test_window']:>10}  "
                  f"{r['n_samples']:>7}  "
                  f"{m.get('precision_mean', 0):>8.4f}  "
                  f"{m.get('recall_mean', 0):>8.4f}  "
                  f"{m.get('f1_mean', 0):>8.4f}  "
                  f"{m.get('balanced_accuracy_mean', 0):>8.4f}  "
                  f"{m.get('mcc_mean', 0):>8.4f}")

print("\n" + "="*80)
print("HOLDOUT VALIDATION COMPLETE - ALL MODELS")
print(f"Total time: {(time.time()-t0_total)/60:.1f} minutes")
print("="*80)

# Final comparison table for holdout (Phase 1, offset 0 F1 score)
print("\n" + "="*80)
print("HOLDOUT VALIDATION COMPARISON TABLE (Phase 1, offset 0 - F1 Score)")
print("="*80)
print(f"{'Model':<25}  {'Phase1_off0':>12}  {'Phase2_off0':>12}  {'Phase3_off0':>12}  {'Phase4_off0':>12}")
print("-" * 70)

for model_name, holdout_results in all_holdout_results.items():
    f1_scores = []
    for phase in [1, 2, 3, 4]:
        if phase in holdout_results:
            nr = holdout_results[phase]['neighbor_results']
            if 0 in nr:
                f1 = nr[0]['metrics'].get('f1_mean', 0)
                f1_scores.append(f"{f1:.4f}")
            else:
                f1_scores.append("N/A")
        else:
            f1_scores.append("N/A")
    print(f"{model_name:<25}  {f1_scores[0]:>12}  {f1_scores[1]:>12}  {f1_scores[2]:>12}  {f1_scores[3]:>12}")

print("="*80)

In [ ]:
# =============================================================================
# CELL 22: PLOT HOLDOUT VALIDATION RESULTS (ALL MODELS)
# =============================================================================

print("\n" + "="*80)
print("GENERATING HOLDOUT VALIDATION FIGURES - ALL MODELS")
print("="*80)

# Make sure all_holdout_results exists from CELL 21
try:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for model_name, holdout_results in all_holdout_results.items():
        print(f"\n{'='*60}")
        print(f"Model: {model_name}")
        print(f"{'='*60}")
        
        # Main holdout bar chart
        print("→ Creating holdout validation bar chart...")
        fig_path = plot_holdout_validation(holdout_results, f"{timestamp}_{model_name.replace(' ', '_')}")
        
        if fig_path:
            print(f"    Figure saved: {fig_path}")
    
    print("\n" + "="*80)
    print("HOLDOUT VALIDATION FIGURES COMPLETE - ALL MODELS")
    print("="*80)
    
except NameError:
    print("Error: all_holdout_results not defined. Run CELL 21 first.")
except Exception as e:
    print(f"Error generating figures: {e}")

In [ ]:
# =============================================================================
# CELL 23: RUN INDEPENDENT TEST (ALL MODELS)
# =============================================================================

print("\n" + "="*80)
print("RUNNING INDEPENDENT TEST EXPERIMENT - ALL MODELS")
print("="*80)

# Get available models
available_models = model_factory.get_available_models()
print(f"\nEvaluating {len(available_models)} models:")
for i, m in enumerate(available_models, 1):
    print(f"  {i}. {m}")
print("="*80)

# Store all independent test results
all_independent_results = {}

t0_total = time.time()

for model_name in available_models:
    print("\n" + "="*80)
    print(f"MODEL: {model_name}")
    print("="*80)
    
    t0 = time.time()
    # REMOVED: sample_ratio parameter - now uses ONLY Phase P data for training
    independent_results = perform_independent_test(model_name, n_repeats=N_REPEATS)
    elapsed = (time.time() - t0) / 60
    print(f"\n  Time: {elapsed:.1f} minutes")
    
    all_independent_results[model_name] = independent_results
    
    # Print summary for this model
    for train_phase in sorted(independent_results.keys()):
        d = independent_results[train_phase]
        tr = d['test_results']
        print(f"\n  Train Phase {train_phase} ({d['train_window']}) → Test Phase {d['test_phase']}:")
        print(f"  {'Label':>6}  {'Window':>10}  {'N_test':>7}  "
              f"{'Prec':>8}  {'Rec':>8}  {'F1':>8}  {'BalAcc':>8}  {'MCC':>8}")
        print("  " + "-"*65)
        
        for label, res in tr.items():
            m = res['metrics']
            print(f"  {label:>6}  {res['test_window']:>10}  "
                  f"{res.get('n_test_samples', 0):>7}  "
                  f"{m.get('precision_mean', 0):>8.4f}  "
                  f"{m.get('recall_mean', 0):>8.4f}  "
                  f"{m.get('f1_mean', 0):>8.4f}  "
                  f"{m.get('balanced_accuracy_mean', 0):>8.4f}  "
                  f"{m.get('mcc_mean', 0):>8.4f}")

print("\n" + "="*80)
print("INDEPENDENT TEST COMPLETE - ALL MODELS")
print(f"Total time: {(time.time()-t0_total)/60:.1f} minutes")
print("="*80)

# Final comparison table for independent test (Phase 1→2, S12 F1 score)
print("\n" + "="*80)
print("INDEPENDENT TEST COMPARISON TABLE (Phase1→2, S12 F1 Score)")
print("="*80)
print(f"{'Model':<25}  {'S12_F1':>10}  {'S22_F1':>10}  {'S32_F1':>10}  {'S42_F1':>10}")
print("-" * 65)

for model_name, independent_results in all_independent_results.items():
    f1_scores = []
    # Phase1→2: S12
    if 1 in independent_results and 'S12' in independent_results[1]['test_results']:
        f1 = independent_results[1]['test_results']['S12']['metrics'].get('f1_mean', 0)
        f1_scores.append(f"{f1:.4f}")
    else:
        f1_scores.append("N/A")
    
    # Phase2→3: S22
    if 2 in independent_results and 'S22' in independent_results[2]['test_results']:
        f1 = independent_results[2]['test_results']['S22']['metrics'].get('f1_mean', 0)
        f1_scores.append(f"{f1:.4f}")
    else:
        f1_scores.append("N/A")
    
    # Phase3→4: S32
    if 3 in independent_results and 'S32' in independent_results[3]['test_results']:
        f1 = independent_results[3]['test_results']['S32']['metrics'].get('f1_mean', 0)
        f1_scores.append(f"{f1:.4f}")
    else:
        f1_scores.append("N/A")
    
    # Phase4→1: S42
    if 4 in independent_results and 'S42' in independent_results[4]['test_results']:
        f1 = independent_results[4]['test_results']['S42']['metrics'].get('f1_mean', 0)
        f1_scores.append(f"{f1:.4f}")
    else:
        f1_scores.append("N/A")
    
    print(f"{model_name:<25}  {f1_scores[0]:>10}  {f1_scores[1]:>10}  {f1_scores[2]:>10}  {f1_scores[3]:>10}")

print("="*80)

In [ ]:
# =============================================================================
# CELL 24: PLOT INDEPENDENT TEST RESULTS (ALL MODELS)
# =============================================================================

print("\n" + "="*80)
print("GENERATING INDEPENDENT TEST FIGURES - ALL MODELS")
print("="*80)

# Make sure all_independent_results exists from CELL 23
try:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for model_name, independent_results in all_independent_results.items():
        print(f"\n{'='*60}")
        print(f"Model: {model_name}")
        print(f"{'='*60}")
        
        print("→ Creating independent test bar chart...")
        fig_path = plot_independent_test(independent_results, f"{timestamp}_{model_name.replace(' ', '_')}")
        
        if fig_path:
            print(f"    Figure saved: {fig_path}")
    
    print("\n" + "="*80)
    print("INDEPENDENT TEST FIGURES COMPLETE - ALL MODELS")
    print("="*80)
    
except NameError:
    print("Error: all_independent_results not defined. Run CELL 23 first.")
except Exception as e:
    print(f"Error generating figures: {e}")

In [ ]:
# =============================================================================
# CELL 25: COMBINED SUMMARY FIGURE (ALL MODELS)
# =============================================================================

print("\n" + "="*80)
print("CREATING COMBINED SUMMARY FIGURES - ALL MODELS")
print("="*80)

# Make sure all results exist from previous cells
try:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for model_name in all_cv_results.keys():
        print(f"\n{'='*60}")
        print(f"Model: {model_name}")
        print(f"{'='*60}")
        
        # Get results for this model
        cv_results = all_cv_results.get(model_name, {})
        holdout_results = all_holdout_results.get(model_name, {})
        independent_results = all_independent_results.get(model_name, {})
        
        if cv_results and holdout_results and independent_results:
            print("Creating Combined 3-experiment summary figure...")
            fig_path = plot_combined_summary(cv_results, holdout_results,
                                             independent_results, timestamp, model_name)
            if fig_path:
                print(f"    Figure saved: {fig_path}")
        else:
            print(f"    Warning: Missing results for {model_name}, skipping figure")
    
    print("\n" + "="*80)
    print("COMBINED SUMMARY FIGURES COMPLETE - ALL MODELS")
    print("="*80)
    
except NameError as e:
    print(f"Error: Required results not defined. {e}")
    print("Make sure CELLS 19, 21, 23 have been run first.")
except Exception as e:
    print(f"Error generating figures: {e}")

In [ ]:
# =============================================================================
# CELL 26: SAVE ALL RESULTS (ALL MODELS)
# =============================================================================

print("\n" + "="*80)
print("SAVING ALL RESULTS - ALL MODELS")
print("="*80)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

for model_name in all_cv_results.keys():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    
    cv_results = all_cv_results.get(model_name, {})
    holdout_results = all_holdout_results.get(model_name, {})
    independent_results = all_independent_results.get(model_name, {})
    
    if cv_results and holdout_results and independent_results:
        json_f, txt_f = save_all_results(cv_results, holdout_results, 
                                         independent_results, timestamp, model_name)
        print(f"    JSON saved: {json_f}")
        print(f"    Text saved: {txt_f}")
    else:
        print(f"    Warning: Missing results for {model_name}, skipping save")

print("\n" + "="*80)
print("ALL RESULTS SAVED - ALL MODELS")
print("="*80)

In [ ]:
# =============================================================================
# CELL 27: PRINT COMPREHENSIVE STATISTICS (ALL MODELS)
# =============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE STATISTICS - ALL MODELS")
print("="*80)

for model_name in all_cv_results.keys():
    print("\n" + "="*80)
    print(f"MODEL: {model_name}")
    print("="*80)
    
    cv_results = all_cv_results.get(model_name, {})
    holdout_results = all_holdout_results.get(model_name, {})
    independent_results = all_independent_results.get(model_name, {})
    
    if cv_results and holdout_results and independent_results:
        print_comprehensive_statistics(cv_results, holdout_results, independent_results)
    else:
        print(f"Warning: Missing results for {model_name}")

print("\n" + "="*80)
print("ALL STATISTICS COMPLETE")
print("="*80)

In [ ]:
!pip install shap --proxy http://127.0.0.1:59527

In [ ]:
!pip install shap rfpimp --proxy http://127.0.0.1:59527

In [ ]:
!pip install rfpimp --proxy http://127.0.0.1:59527

In [ ]:
# =============================================================================
# CELL 27: FEATURE IMPORTANCE — IMPORTS AND SETUP
# =============================================================================

import shap
from sklearn.inspection import permutation_importance as sk_permutation_importance

try:
    from rfpimp import (importances as rfpimp_importances, plot_importances,
                        stemplot_importances, plot_corr_heatmap,
                        feature_dependence_matrix, plot_dependence_heatmap)
    RFPIMP_AVAILABLE = True
    print("rfpimp available")
except ImportError:
    RFPIMP_AVAILABLE = False
    print("rfpimp NOT available - install with: pip install rfpimp")

CATEGORY_COLORS = {
    'morphological': '#D7191C',
    'shape':         '#2166AC',
    'biological':    '#1A9641',
    'motion':        '#FF7F00',
    'intensity':     '#7B2D8B',
}

FEATURE_DISPLAY_NAMES = {
    'Area':                           'Area',
    'Volume':                         'Volume',
    'Perimeter':                      'Perimeter',
    'Compactness':                    'Compactness',
    'Sphericity':                     'Sphericity',
    'Solidity':                       'Solidity',
    'Extent':                         'Extent',
    'Ellipsoid-Prolate':              'Ellipsoid-Prolate',
    'Ellipsoid-Oblate':               'Ellipsoid-Oblate',
    'Nucleus-Cytoplasm Volume Ratio':  'NC Volume Ratio',
    'Speed':                          'Speed',
    'Displacement':                   'Displacement',
    'Intensity-Mean':                 'Intensity-Mean',
    'Intensity-Sum':                  'Intensity-Sum',
    'Intensity-StdDev':               'Intensity-StdDev',
    'Intensity-Max':                  'Intensity-Max',
    'Intensity-Min':                  'Intensity-Min',
}

FEATURE_CUSTOM_COLORS = {
    'Area':                           'red',
    'Volume':                         'darkred',
    'Perimeter':                      'crimson',
    'Compactness':                    'blue',
    'Sphericity':                     'green',
    'Solidity':                       'royalblue',
    'Extent':                         'navy',
    'Ellipsoid-Prolate':              'steelblue',
    'Ellipsoid-Oblate':               'dodgerblue',
    'Nucleus-Cytoplasm Volume Ratio':  'orange',
    'Speed':                          'indigo',
    'Displacement':                   'purple',
    'Intensity-Mean':                 'cyan',
    'Intensity-Sum':                  'pink',
    'Intensity-StdDev':               'magenta',
    'Intensity-Max':                  'olive',
    'Intensity-Min':                  'teal',
}

def _feature_category(feat_name):
    for cat, feats in FEATURE_CATEGORIES.items():
        if feat_name in feats:
            return cat
    return 'other'

def _feature_color(feat_name):
    return CATEGORY_COLORS.get(_feature_category(feat_name), '#555555')

CATEGORY_PATCHES = [
    mpatches.Patch(color=CATEGORY_COLORS[c], label=c.capitalize())
    for c in ['morphological', 'shape', 'biological', 'motion', 'intensity']
]

pd.set_option('display.precision', 3)

In [ ]:
# =============================================================================
# CELL 28: TRAIN MODELS FOR FEATURE IMPORTANCE (ALL 4 PHASES)
# GB-LGBM VotingClassifier + RandomForest (for rfpimp)
# =============================================================================

print("\n" + "="*80)
print("CELL 28: TRAINING MODELS FOR FEATURE IMPORTANCE ANALYSIS")
print("="*80)

FI_DATA    = {}
SHAP_STORE = {}

for phase in [1, 2, 3, 4]:
    cfg         = BEST_CONFIGS[phase]
    start_frame = cfg['start_frame']
    wl          = cfg['window_length']
    end_frame   = start_frame + wl - 1

    print(f"\nPhase {phase}: frames {start_frame}-{end_frame} ...", end=" ", flush=True)

    win = feature_processor.create_time_window_data(start_frame, wl)
    if win is None:
        print("No data - skipped.")
        continue

    X, y       = win['X'], win['y']
    feat_names = win['features']
    disp_names = [FEATURE_DISPLAY_NAMES.get(f, f) for f in feat_names]
    X_df       = pd.DataFrame(X, columns=feat_names)
    X_df_disp  = pd.DataFrame(X, columns=disp_names)

    # Primary model: GB-LGBM VotingClassifier
    vc = model_factory.create_gb_lgbm_hybrid(random_state=42)
    vc.fit(X, y)
    lgbm_model = vc.named_estimators_['lgbm']
    gb_model   = vc.named_estimators_['gb']

    # RandomForest used by rfpimp importances
    rf_model = RandomForestClassifier(
        criterion='entropy', max_depth=40, min_samples_split=8,
        n_estimators=500, random_state=42)
    rf_model.fit(X, y)

    FI_DATA[phase] = {
        'X':             X,
        'y':             y,
        'X_df':          X_df,
        'X_df_disp':     X_df_disp,
        'features':      feat_names,
        'display_names': disp_names,
        'n_samples':     win['n_samples'],
        'normal_cells':  win['normal_cells'],
        'ips_cells':     win['ips_cells'],
        'model_vc':      vc,
        'model_lgbm':    lgbm_model,
        'model_gb':      gb_model,
        'model_rf':      rf_model,
    }
    print(f"Done. N={win['n_samples']} (Normal={win['normal_cells']}, iPS={win['ips_cells']})")

print("\n" + "="*80)
print(f"Models trained for {len(FI_DATA)} phases.")
print("="*80)

In [ ]:
# =============================================================================
# CELL 29: CORRELATION HEATMAPS — ALL 4 PHASES (inferno, viridis, diverging)
# =============================================================================

print("\n" + "="*80)
print("CELL 29: CORRELATION HEATMAPS - ALL 4 PHASES")
print("="*80)

colormaps_to_plot = [
    ('inferno',   'inferno'),
    ('viridis',   'viridis'),
    ('diverging',  sns.diverging_palette(220, 10, as_cmap=True)),
]

for cmap_name, cmap in colormaps_to_plot:
    fig, axes = plt.subplots(2, 2, figsize=(22, 18))
    axes_flat = axes.flatten()

    for idx, phase in enumerate([1, 2, 3, 4]):
        if phase not in FI_DATA:
            continue
        ax   = axes_flat[idx]
        d    = FI_DATA[phase]
        corr = d['X_df_disp'].corr(method='pearson')

        sns.heatmap(
            corr,
            ax=ax,
            annot=True,
            linewidths=0.2,
            cmap=cmap,
            fmt='.2f',
            annot_kws={'size': 8},
            square=True,
        )
        cfg = BEST_CONFIGS[phase]
        ax.set_title(
            f'Phase {phase} - Pearson Correlation Matrix\n'
            f'Frames {cfg["start_frame"]}-{cfg["start_frame"] + WINDOW_LENGTH - 1}  '
            f'(N={d["n_samples"]})',
            fontweight='bold', fontsize=12, pad=12)
        ax.tick_params(axis='x', rotation=45, labelsize=8)
        ax.tick_params(axis='y', rotation=0,  labelsize=8)

    fig.suptitle(f'Pearson Correlation Matrix - {cmap_name} colormap',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    fname = os.path.join(PLOTS_DIR,
                         f'correlation_heatmap_{cmap_name}_{TIMESTAMP}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Correlation heatmap ({cmap_name}) saved: {fname}")
    plt.show()
    plt.close(fig)

In [ ]:
# =============================================================================
# CELL 30: FEATURE HISTOGRAMS — ALL 4 PHASES
# =============================================================================

print("\n" + "="*80)
print("CELL 30: FEATURE HISTOGRAMS - ALL 4 PHASES")
print("="*80)

for phase in [1, 2, 3, 4]:
    if phase not in FI_DATA:
        continue
    d   = FI_DATA[phase]
    cfg = BEST_CONFIGS[phase]

    d['X_df_disp'].hist(bins=25, figsize=(22, 22))
    plt.suptitle(
        f'Feature Distributions - Phase {phase}  '
        f'Frames {cfg["start_frame"]}-{cfg["start_frame"] + WINDOW_LENGTH - 1}  '
        f'(N={d["n_samples"]})',
        fontsize=14, fontweight='bold')
    plt.tight_layout()
    fname = os.path.join(PLOTS_DIR,
                         f'feature_histograms_phase{phase}_{TIMESTAMP}.png')
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Feature histograms Phase {phase} saved: {fname}")
    plt.show()
    plt.close('all')


In [ ]:
# =============================================================================
# CELL 31: rfpimp IMPORTANCES, STEMPLOT, CORR HEATMAP, DEPENDENCE HEATMAP
# Uses GB-LGBM VotingClassifier (RF removed)
# =============================================================================

def _apply_thin_box_spines(fig=None):
    """Add thin top and right spines to all axes in the current/given figure,
    matching _set_ax_style used in the CV figures."""
    if fig is None:
        fig = plt.gcf()
    for ax in fig.axes:
        ax.spines['top'].set_visible(True)
        ax.spines['top'].set_color('#aaaaaa')
        ax.spines['top'].set_linewidth(0.6)
        ax.spines['right'].set_visible(True)
        ax.spines['right'].set_color('#aaaaaa')
        ax.spines['right'].set_linewidth(0.6)


print("\n" + "="*80)
print("CELL 31: rfpimp ANALYSIS - ALL 4 PHASES  (GB-LGBM VotingClassifier)")
print("="*80)

if RFPIMP_AVAILABLE:
    for phase in [1, 2, 3, 4]:
        if phase not in FI_DATA:
            continue
        d        = FI_DATA[phase]
        X_df     = d['X_df']
        y_series = pd.Series(d['y'])
        vc_model = d['model_vc']          # GB-LGBM VotingClassifier (RF removed)
        cfg      = BEST_CONFIGS[phase]

        print(f"\nPhase {phase}: computing rfpimp importances (GB-LGBM) ...",
              end=" ", flush=True)
        I = rfpimp_importances(vc_model, X_df, y_series)
        print("done.")
        print(I)

        # ── plot_importances default ──────────────────────────────────────────
        plot_importances(I)
        _apply_thin_box_spines()
        plt.title(
            f'Phase {phase} - Feature Importances (rfpimp, GB-LGBM)\n'
            f'Frames {cfg["start_frame"]}-{cfg["start_frame"] + WINDOW_LENGTH - 1}',
            fontweight='bold')
        plt.tight_layout()
        fname = os.path.join(PLOTS_DIR,
                             f'rfpimp_importances_phase{phase}_{TIMESTAMP}.png')
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"rfpimp importances Phase {phase} saved: {fname}")
        plt.show()
        plt.close('all')

        # ── plot_importances gold ─────────────────────────────────────────────
        plot_importances(I, color='#FDDB7D')
        _apply_thin_box_spines()
        plt.title(
            f'Phase {phase} - Feature Importances (rfpimp, gold, GB-LGBM)\n'
            f'Frames {cfg["start_frame"]}-{cfg["start_frame"] + WINDOW_LENGTH - 1}',
            fontweight='bold')
        plt.tight_layout()
        fname = os.path.join(PLOTS_DIR,
                             f'rfpimp_importances_gold_phase{phase}_{TIMESTAMP}.png')
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"rfpimp importances (gold) Phase {phase} saved: {fname}")
        plt.show()
        plt.close('all')

        # ── stemplot ──────────────────────────────────────────────────────────
        stemplot_importances(I, vscale=.7)
        _apply_thin_box_spines()
        plt.title(
            f'Phase {phase} - Feature Importances Stemplot (GB-LGBM)\n'
            f'Frames {cfg["start_frame"]}-{cfg["start_frame"] + WINDOW_LENGTH - 1}',
            fontweight='bold')
        plt.tight_layout()
        fname = os.path.join(PLOTS_DIR,
                             f'rfpimp_stemplot_phase{phase}_{TIMESTAMP}.png')
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"rfpimp stemplot Phase {phase} saved: {fname}")
        plt.show()
        plt.close('all')

        # ── correlation heatmap (no model needed) ─────────────────────────────
        viz = plot_corr_heatmap(X_df, figsize=(16, 16))
        fname = os.path.join(PLOTS_DIR,
                             f'rfpimp_corr_heatmap_phase{phase}_{TIMESTAMP}.png')
        viz.save(fname)
        print(f"rfpimp corr heatmap Phase {phase} saved: {fname}")

        # ── feature dependence matrix (no model needed) ───────────────────────
        D = feature_dependence_matrix(X_df, sort_by_dependence=True)
        print(f"\nPhase {phase} - Feature Dependence (sorted):")
        print(D['Dependence'].sort_values(ascending=False))

        viz = plot_dependence_heatmap(D, figsize=(18, 18))
        fname = os.path.join(PLOTS_DIR,
                             f'rfpimp_dependence_heatmap_phase{phase}_{TIMESTAMP}.png')
        viz.save(fname)
        print(f"rfpimp dependence heatmap Phase {phase} saved: {fname}")

else:
    print("rfpimp not available - skipping Cell 31. Install with: pip install rfpimp")

In [ ]:
# =============================================================================
# CELL 32: PERMUTATION IMPORTANCE — GB-LGBM ONLY, ALL 4 PHASES
# (RandomForest removed)
# =============================================================================

print("\n" + "="*80)
print("CELL 32: PERMUTATION IMPORTANCE - GB-LGBM ONLY, ALL 4 PHASES")
print("="*80)

for phase in [1, 2, 3, 4]:
    if phase not in FI_DATA:
        continue
    d          = FI_DATA[phase]
    X, y       = d['X'], d['y']
    disp_names = d['display_names']
    cfg        = BEST_CONFIGS[phase]

    # Only GB-LGBM (RandomForest removed)
    for model_name, model in [('GB-LGBM', d['model_vc'])]:

        print(f"  Phase {phase} - {model_name}: permutation importance ...",
              end=" ", flush=True)
        result     = sk_permutation_importance(
            model, X, y, n_repeats=5, random_state=42, n_jobs=2)
        sorted_idx = result.importances_mean.argsort()
        print("done.")

        fig, ax = plt.subplots(figsize=(12, 12))
        ax.boxplot(
            result.importances[sorted_idx].T,
            vert=False,
            labels=[disp_names[i] for i in sorted_idx])

        # ── thin top/right spines matching _set_ax_style ─────────────────────
        ax.spines['left'].set_visible(True)
        ax.spines['left'].set_color('#222222')
        ax.spines['left'].set_linewidth(1.4)
        ax.spines['bottom'].set_visible(True)
        ax.spines['bottom'].set_color('#222222')
        ax.spines['bottom'].set_linewidth(1.4)
        ax.spines['top'].set_visible(True)
        ax.spines['top'].set_color('#aaaaaa')
        ax.spines['top'].set_linewidth(0.6)
        ax.spines['right'].set_visible(True)
        ax.spines['right'].set_color('#aaaaaa')
        ax.spines['right'].set_linewidth(0.6)

        ax.set_title(
            f'Phase {phase} - Permutation Importances ({model_name})\n'
            f'Frames {cfg["start_frame"]}-{cfg["start_frame"] + WINDOW_LENGTH - 1}',
            fontsize=15, fontweight='bold')
        fig.tight_layout()
        plt.xlabel('\nPermutation accuracy feature importance',
                   fontsize=15, fontweight='bold')
        plt.ylabel('\nFeatures', fontsize=15, fontweight='bold')
        plt.xticks(fontsize=15)
        plt.yticks(fontsize=12)

        fname = os.path.join(
            PLOTS_DIR,
            f'permutation_importance_{model_name.lower().replace("-", "_")}'
            f'_phase{phase}_{TIMESTAMP}.png')
        fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"  Permutation importance ({model_name}) Phase {phase} saved: {fname}")
        plt.show()
        plt.close(fig)

In [ ]:
# =============================================================================
# CELL 33: SHAP SUMMARY PLOTS — BEESWARM, BAR, BAR WITH CUSTOM COLORS
#          LGBM TreeExplainer, ALL 4 PHASES
# =============================================================================

print("\n" + "="*80)
print("CELL 33: SHAP SUMMARY PLOTS - ALL 4 PHASES")
print("="*80)

for phase in [1, 2, 3, 4]:
    if phase not in FI_DATA:
        continue
    d          = FI_DATA[phase]
    X          = d['X']
    feat_names = d['features']
    disp_names = d['display_names']
    lgbm_model = d['model_lgbm']
    X_df_disp  = d['X_df_disp']
    cfg        = BEST_CONFIGS[phase]

    print(f"  Phase {phase}: LGBM SHAP values ...", end=" ", flush=True)
    explainer_lgbm  = shap.TreeExplainer(lgbm_model)
    shap_values_raw = explainer_lgbm.shap_values(X)
    if isinstance(shap_values_raw, list):
        sv = shap_values_raw[1]
    else:
        sv = shap_values_raw
    print("done.")

    SHAP_STORE[phase] = {
        'shap_values':   sv,
        'X':             X,
        'X_df_disp':     X_df_disp,
        'features':      feat_names,
        'display_names': disp_names,
        'explainer':     explainer_lgbm,
    }

    # Beeswarm / dot summary
    shap.summary_plot(sv, X_df_disp, feature_names=disp_names,
                      show=False, max_display=len(feat_names))
    plt.title(
        f'Phase {phase} - Local explanation summary (LGBM TreeExplainer)\n'
        f'Frames {cfg["start_frame"]}-{cfg["start_frame"] + WINDOW_LENGTH - 1}')
    fname = os.path.join(PLOTS_DIR,
                         f'shap_summary_beeswarm_phase{phase}_{TIMESTAMP}.png')
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  SHAP beeswarm Phase {phase} saved: {fname}")
    plt.show()
    plt.close('all')

    # Bar chart — viridis colormap
    shap.summary_plot(sv, X_df_disp, plot_type='bar',
                      color=plt.cm.viridis.colors,
                      show=False)
    plt.title(f'Phase {phase} - Global feature importance')
    fname = os.path.join(PLOTS_DIR,
                         f'shap_bar_viridis_phase{phase}_{TIMESTAMP}.png')
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  SHAP bar (viridis) Phase {phase} saved: {fname}")
    plt.show()
    plt.close('all')

    # Bar chart — custom per-feature colors
    feat_color_list = [FEATURE_CUSTOM_COLORS.get(f, 'gray') for f in feat_names]
    shap.summary_plot(sv, X_df_disp, plot_type='bar',
                      color=feat_color_list,
                      show=False)
    plt.title(f'Phase {phase} - Global feature importance (custom colors)')
    fname = os.path.join(PLOTS_DIR,
                         f'shap_bar_custom_colors_phase{phase}_{TIMESTAMP}.png')
    plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  SHAP bar (custom colors) Phase {phase} saved: {fname}")
    plt.show()
    plt.close('all')

In [ ]:
# =============================================================================
# CELL 34: SHAP DEPENDENCE PLOTS — ALL 17 FEATURES x ALL 4 PHASES
#          Auto interaction coloring AND self interaction_index
# =============================================================================

print("\n" + "="*80)
print("CELL 34: SHAP DEPENDENCE PLOTS - ALL 17 FEATURES x ALL 4 PHASES")
print("="*80)

for phase in [1, 2, 3, 4]:
    if phase not in SHAP_STORE:
        continue
    s          = SHAP_STORE[phase]
    sv         = s['shap_values']
    X          = s['X']
    feat_names = s['features']
    disp_names = s['display_names']
    cfg        = BEST_CONFIGS[phase]

    # Individual dependence plots — auto interaction coloring
    for feat, dname in zip(feat_names, disp_names):
        shap.dependence_plot(
            feat, sv, X,
            feature_names=feat_names,
            show=False)
        plt.title(
            f'Phase {phase} - SHAP Dependence: {dname}\n'
            f'Frames {cfg["start_frame"]}-{cfg["start_frame"] + WINDOW_LENGTH - 1}')
        fname = os.path.join(
            PLOTS_DIR,
            f'shap_dep_{dname.replace(" ","_").replace("/","_")}'
            f'_phase{phase}_{TIMESTAMP}.png')
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close('all')

    # Individual dependence plots — self as interaction_index
    for feat, dname in zip(feat_names, disp_names):
        shap.dependence_plot(
            feat, sv, X,
            feature_names=feat_names,
            interaction_index=feat,
            show=False)
        plt.title(
            f'Phase {phase} - SHAP Dependence (self-interaction): {dname}\n'
            f'Frames {cfg["start_frame"]}-{cfg["start_frame"] + WINDOW_LENGTH - 1}')
        fname = os.path.join(
            PLOTS_DIR,
            f'shap_dep_self_{dname.replace(" ","_").replace("/","_")}'
            f'_phase{phase}_{TIMESTAMP}.png')
        plt.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
        plt.close('all')

    print(f"  Phase {phase}: {len(feat_names)} dependence plots saved.")

In [ ]:
# =============================================================================
# CELL 34: SHAP DEPENDENCE PLOTS — ALL 17 FEATURES x ALL 4 PHASES
#          Combined: 1 figure per feature, 4 phases as columns
#          Variant A: Auto interaction coloring
#          Variant B: Self interaction_index
# =============================================================================

print("\n" + "="*80)
print("CELL 34: SHAP DEPENDENCE PLOTS - ALL 17 FEATURES x ALL 4 PHASES (COMBINED)")
print("="*80)
print("Layout: 1 figure per feature × 2 variants (auto / self) = 34 total figures")
print("Each figure: 1 row × 4 columns  (Phase 1 | Phase 2 | Phase 3 | Phase 4)")
print("="*80)

# ── Collect data for all phases first ────────────────────────────────────────
phase_data = {}
for phase in [1, 2, 3, 4]:
    if phase not in SHAP_STORE:
        continue
    s = SHAP_STORE[phase]
    phase_data[phase] = {
        'sv':         s['shap_values'],
        'X':          s['X'],
        'feat_names': s['features'],
        'disp_names': s['display_names'],
        'cfg':        BEST_CONFIGS[phase],
    }

# Use feature list from Phase 1 (all phases share the same 17 features)
ref_feat_names = phase_data[min(phase_data.keys())]['feat_names']
ref_disp_names = phase_data[min(phase_data.keys())]['disp_names']
available_phases = sorted(phase_data.keys())

# ── Helper: apply thin top/right spines ──────────────────────────────────────
def _thin_box(ax):
    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_color('#222222')
    ax.spines['left'].set_linewidth(1.4)
    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_color('#222222')
    ax.spines['bottom'].set_linewidth(1.4)
    ax.spines['top'].set_visible(True)
    ax.spines['top'].set_color('#aaaaaa')
    ax.spines['top'].set_linewidth(0.6)
    ax.spines['right'].set_visible(True)
    ax.spines['right'].set_color('#aaaaaa')
    ax.spines['right'].set_linewidth(0.6)


# ── VARIANT A: Auto interaction coloring ─────────────────────────────────────
print("\nVariant A: Auto interaction coloring")
print("-"*60)

for feat, dname in zip(ref_feat_names, ref_disp_names):

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle(
        f'SHAP Dependence: {dname}  (auto interaction coloring)\n'
        f'All 4 Phases — Window length {WINDOW_LENGTH} frames',
        fontsize=13, fontweight='bold', y=1.02)

    for col_idx, phase in enumerate(available_phases):
        ax  = axes[col_idx]
        d   = phase_data[phase]
        cfg = d['cfg']

        # SHAP draws onto the current matplotlib axes; we need to capture it
        plt.sca(ax)
        shap.dependence_plot(
            feat,
            d['sv'],
            d['X'],
            feature_names=d['feat_names'],
            ax=ax,
            show=False)

        _thin_box(ax)
        ax.set_title(
            f'Phase {phase}\n'
            f'Frames {cfg["start_frame"]}–{cfg["start_frame"] + WINDOW_LENGTH - 1}',
            fontsize=11, fontweight='bold')

    plt.tight_layout()
    safe_name = dname.replace(' ', '_').replace('/', '_')
    fname = os.path.join(PLOTS_DIR,
                         f'shap_dep_auto_{safe_name}_{TIMESTAMP}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  Saved (auto)  : {safe_name}")
    plt.show()
    plt.close(fig)

print(f"\nVariant A complete — {len(ref_feat_names)} combined figures saved.")


# ── VARIANT B: Self interaction_index ────────────────────────────────────────
print("\nVariant B: Self interaction_index")
print("-"*60)

for feat, dname in zip(ref_feat_names, ref_disp_names):

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle(
        f'SHAP Dependence: {dname}  (self-interaction)\n'
        f'All 4 Phases — Window length {WINDOW_LENGTH} frames',
        fontsize=13, fontweight='bold', y=1.02)

    for col_idx, phase in enumerate(available_phases):
        ax  = axes[col_idx]
        d   = phase_data[phase]
        cfg = d['cfg']

        plt.sca(ax)
        shap.dependence_plot(
            feat,
            d['sv'],
            d['X'],
            feature_names=d['feat_names'],
            interaction_index=feat,
            ax=ax,
            show=False)

        _thin_box(ax)
        ax.set_title(
            f'Phase {phase}\n'
            f'Frames {cfg["start_frame"]}–{cfg["start_frame"] + WINDOW_LENGTH - 1}',
            fontsize=11, fontweight='bold')

    plt.tight_layout()
    safe_name = dname.replace(' ', '_').replace('/', '_')
    fname = os.path.join(PLOTS_DIR,
                         f'shap_dep_self_{safe_name}_{TIMESTAMP}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  Saved (self)  : {safe_name}")
    plt.show()
    plt.close(fig)

print(f"\nVariant B complete — {len(ref_feat_names)} combined figures saved.")
print(f"\nCell 34 total: {len(ref_feat_names) * 2} combined figures "
      f"(was {len(ref_feat_names) * len(available_phases) * 2} individual plots).")

In [ ]:
# =============================================================================
# CELL 35: SHAP INTERACTION VALUES — ALL 4 PHASES  (COMBINED 2×2 FIGURE)
# Model  : FI_DATA[phase]['model_lgbm'] — original LightGBM estimator
# Sampling: X subsampled to MAX_INTERACT_SAMPLES rows — interaction values are
#           O(n_features² × n_samples); 300 balanced rows are sufficient.
# =============================================================================

import io
from PIL import Image

MAX_INTERACT_SAMPLES = 300   # ← tune up/down; 300 ≈ <2 min total for 4 phases

print("\n" + "="*80)
print("CELL 35: SHAP INTERACTION VALUES - ALL 4 PHASES  (combined 2×2)")
print(f"         Subsampled to {MAX_INTERACT_SAMPLES} rows per phase for speed")
print("="*80)

def _balanced_subsample(X, y, n, seed=42):
    """Return at most n rows with equal class representation."""
    rng    = np.random.RandomState(seed)
    idx0   = np.where(y == 0)[0]
    idx1   = np.where(y == 1)[0]
    n_each = min(n // 2, len(idx0), len(idx1))
    chosen = np.concatenate([
        rng.choice(idx0, n_each, replace=False),
        rng.choice(idx1, n_each, replace=False),
    ])
    rng.shuffle(chosen)
    return X[chosen], y[chosen], chosen


# ── Step 1: compute interaction values & render each phase to a buffer ────────
phase_buffers = {}

for phase in [1, 2, 3, 4]:
    if phase not in SHAP_STORE:
        continue

    s         = SHAP_STORE[phase]
    X_full    = s['X']
    y_full    = FI_DATA[phase]['y']
    X_df_disp = s['X_df_disp']
    cfg       = BEST_CONFIGS[phase]

    # original LightGBM estimator
    lgbm_model = FI_DATA[phase]['model_lgbm']

    # ── subsample ────────────────────────────────────────────────────────────
    X_sub, y_sub, sub_idx = _balanced_subsample(
        X_full, y_full, MAX_INTERACT_SAMPLES, seed=42)
    X_df_sub = X_df_disp.iloc[sub_idx].reset_index(drop=True)

    print(f"  Phase {phase}: SHAP interaction values "
          f"(n={len(X_sub)}) ...", end=" ", flush=True)
    explainer = shap.TreeExplainer(lgbm_model)
    siv_raw   = explainer.shap_interaction_values(X_sub)
    siv       = siv_raw[1] if isinstance(siv_raw, list) else siv_raw
    print("done.")

    # stored for use in Cell 36
    SHAP_STORE[phase]['shap_interaction_values'] = siv
    SHAP_STORE[phase]['shap_interaction_X_df']   = X_df_sub

    # ── render to buffer ─────────────────────────────────────────────────────
    shap.summary_plot(siv, X_df_sub, plot_type='interaction', show=False)
    plt.scatter([], []).set_sizes([10])
    plt.gcf().set_size_inches(10, 8)
    plt.title(
        f'Phase {phase} — SHAP Interaction Summary\n'
        f'Frames {cfg["start_frame"]}–{cfg["start_frame"] + WINDOW_LENGTH - 1}'
        f'  (n={len(X_sub)})',
        fontsize=12, fontweight='bold')
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, dpi=150, bbox_inches='tight', facecolor='white')
    buf.seek(0)
    phase_buffers[phase] = buf
    plt.close('all')
    print(f"  Phase {phase}: rendered to buffer.")


# ── Step 2: assemble 2×2 combined figure ─────────────────────────────────────
phases_ordered = [p for p in [1, 2, 3, 4] if p in phase_buffers]

if phases_ordered:
    fig, axes = plt.subplots(2, 2, figsize=(22, 18))
    axes_flat = axes.flatten()

    for idx, phase in enumerate(phases_ordered):
        ax  = axes_flat[idx]
        img = Image.open(phase_buffers[phase])
        ax.imshow(np.array(img))
        ax.axis('off')
        ax.spines['top'].set_visible(True);    ax.spines['top'].set_color('#aaaaaa');    ax.spines['top'].set_linewidth(0.6)
        ax.spines['right'].set_visible(True);  ax.spines['right'].set_color('#aaaaaa');  ax.spines['right'].set_linewidth(0.6)
        ax.spines['left'].set_visible(True);   ax.spines['left'].set_color('#222222');   ax.spines['left'].set_linewidth(1.4)
        ax.spines['bottom'].set_visible(True); ax.spines['bottom'].set_color('#222222'); ax.spines['bottom'].set_linewidth(1.4)

    for idx in range(len(phases_ordered), 4):
        axes_flat[idx].axis('off')

    fig.suptitle(
        f'SHAP Interaction Summary — All 4 Phases\n'
        f'Window {WINDOW_LENGTH} frames  |  GB-LGBM (LGBM TreeExplainer)'
        f'  |  n={MAX_INTERACT_SAMPLES} rows/phase',
        fontsize=15, fontweight='bold', y=1.01)

    plt.tight_layout()
    fname = os.path.join(PLOTS_DIR,
                         f'shap_interaction_summary_combined_{TIMESTAMP}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"\n  Combined 2×2 figure saved: {fname}")
    plt.show()
    plt.close(fig)

    for buf in phase_buffers.values():
        buf.close()

print("\nCell 35 complete.")

In [ ]:
# =============================================================================
# CELL 36: SHAP FEATURE PAIR INTERACTION DEPENDENCE — ALL 4 PHASES
# Pairs: NC Volume Ratio x Intensity-Min  |  Area x Intensity-Min
# Layout: 1 figure per pair, 4 phases as columns (1 row × 4)
# Note: uses the subsampled X stored by Cell 35 (shap_interaction_X_df)
# =============================================================================

print("\n" + "="*80)
print("CELL 36: SHAP FEATURE PAIR INTERACTION DEPENDENCE - ALL 4 PHASES")
print("="*80)

INTERACTION_PAIRS = [
    ('Nucleus-Cytoplasm Volume Ratio', 'Intensity-Min'),
    ('Area',                           'Intensity-Min'),
]

available_phases = [p for p in [1, 2, 3, 4]
                    if p in SHAP_STORE
                    and 'shap_interaction_values' in SHAP_STORE[p]]

for feat1, feat2 in INTERACTION_PAIRS:

    # resolve display names from first available phase
    ref        = SHAP_STORE[available_phases[0]]
    feat_names = ref['features']
    disp_names = ref['display_names']

    if feat1 not in feat_names or feat2 not in feat_names:
        print(f"  Skipping pair ({feat1} x {feat2}): feature not found.")
        continue

    idx1 = feat_names.index(feat1)
    idx2 = feat_names.index(feat2)
    d1   = disp_names[idx1]
    d2   = disp_names[idx2]

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    fig.suptitle(
        f'SHAP Interaction: {d1} vs {d2}\n'
        f'All 4 Phases — Window length {WINDOW_LENGTH} frames',
        fontsize=13, fontweight='bold', y=1.02)

    for col_idx, phase in enumerate(available_phases):
        s        = SHAP_STORE[phase]
        siv      = s['shap_interaction_values']
        cfg      = BEST_CONFIGS[phase]

        # use the subsampled X that matches siv row count
        X_df_sub = s['shap_interaction_X_df']
        X_sub    = X_df_sub.values

        ax = axes[col_idx]
        plt.sca(ax)
        shap.dependence_plot(
            [idx1, idx2],
            siv,
            X_sub,
            feature_names=disp_names,
            ax=ax,
            show=False)

        ax.spines['left'].set_color('#222222');   ax.spines['left'].set_linewidth(1.4)
        ax.spines['bottom'].set_color('#222222'); ax.spines['bottom'].set_linewidth(1.4)
        ax.spines['top'].set_visible(True);       ax.spines['top'].set_color('#aaaaaa');   ax.spines['top'].set_linewidth(0.6)
        ax.spines['right'].set_visible(True);     ax.spines['right'].set_color('#aaaaaa'); ax.spines['right'].set_linewidth(0.6)

        ax.set_title(
            f'Phase {phase}\n'
            f'Frames {cfg["start_frame"]}–{cfg["start_frame"] + WINDOW_LENGTH - 1}',
            fontsize=11, fontweight='bold')

    plt.tight_layout()
    safe1 = d1.replace(' ', '_').replace('/', '_')
    safe2 = d2.replace(' ', '_').replace('/', '_')
    fname = os.path.join(PLOTS_DIR,
                         f'shap_pair_{safe1}_x_{safe2}_{TIMESTAMP}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  Saved: {safe1} x {safe2}")
    plt.show()
    plt.close(fig)

print("\nCell 36 complete.")

In [ ]:
# =============================================================================
# CELL 37: COMBINED FEATURE IMPORTANCE COMPARISON + SAVE RESULTS
# LGBM Gain | Mean |SHAP| | GB-LGBM Permutation
# (RandomForest removed throughout)
# =============================================================================

def _lgbm_gain_importance(lgbm_model, feat_names):
    booster   = lgbm_model.booster_
    gain_dict = dict(zip(booster.feature_name(),
                         booster.feature_importance(importance_type='gain')))
    return np.array([gain_dict.get(f, 0.0) for f in feat_names], dtype=np.float64)


def plot_combined_feature_importance(fi_data, shap_store, timestamp):
    font_family = setup_publication_fonts()
    rc = _base_rcparams(font_family)
    rc.update({'xtick.labelsize': 9, 'ytick.labelsize': 10,
               'axes.titlesize': 11, 'legend.fontsize': 9})
    plt.rcParams.update(rc)

    phases    = sorted(fi_data.keys())
    n_phases  = len(phases)

    # 3 columns — RF removed
    fig, axes = plt.subplots(n_phases, 3, figsize=(21, 5.5 * n_phases),
                              constrained_layout=True)
    if n_phases == 1:
        axes = axes[np.newaxis, :]

    col_titles = [
        'LGBM Gain Importance',
        'Mean |SHAP Value| (LGBM)',
        'Permutation Importance (F1)\nGB-LGBM VotingClassifier',
    ]

    fi_numeric = {}

    for row, phase in enumerate(phases):
        d          = fi_data[phase]
        feat_names = d['features']
        disp_names = d['display_names']
        f_colors   = [_feature_color(f) for f in feat_names]
        X, y       = d['X'], d['y']
        cfg        = BEST_CONFIGS[phase]

        # LGBM gain
        gain_vals = _lgbm_gain_importance(d['model_lgbm'], feat_names)
        gain_vals = gain_vals / (gain_vals.sum() + 1e-12)

        # Mean |SHAP|
        sv        = shap_store[phase]['shap_values'] if phase in shap_store \
                    else np.zeros(len(feat_names))
        shap_vals = np.abs(sv).mean(axis=0) if sv.ndim == 2 \
                    else np.zeros(len(feat_names))

        # GB-LGBM permutation only (RF removed)
        print(f"  Phase {phase}: permutation importance (GB-LGBM) ...",
              end=" ", flush=True)
        perm_gblgbm = sk_permutation_importance(
            d['model_vc'], X, y, n_repeats=5,
            random_state=42, n_jobs=-1, scoring='f1')
        print("done.")

        fi_numeric[phase] = {
            'features':         feat_names,
            'gain':             gain_vals.tolist(),
            'shap_mean':        shap_vals.tolist(),
            'perm_gblgbm_mean': perm_gblgbm.importances_mean.tolist(),
        }

        all_vals = [gain_vals, shap_vals, perm_gblgbm.importances_mean]

        for col, vals in enumerate(all_vals):
            ax       = axes[row, col]
            order    = np.argsort(vals)
            o_vals   = vals[order]
            o_names  = [disp_names[i] for i in order]
            o_colors = [f_colors[i]   for i in order]

            bars = ax.barh(range(len(o_vals)), o_vals,
                           color=o_colors, edgecolor='white', linewidth=0.4)

            vmax = max(abs(o_vals).max(), 1e-9)
            for bi, (bar, val) in enumerate(zip(bars, o_vals)):
                ax.text(val + vmax * 0.01, bi,
                        f'{val:.4f}', va='center', fontsize=8)

            ax.set_yticks(range(len(o_names)))
            ax.set_yticklabels(o_names, fontsize=9)
            ax.set_xlabel('Importance', fontsize=10)

            if row == 0:
                ax.set_title(col_titles[col], fontweight='bold', fontsize=11, pad=8)
            if col == 0:
                ax.set_ylabel(
                    f'Phase {phase}\n(frames {cfg["start_frame"]}-'
                    f'{cfg["start_frame"] + WINDOW_LENGTH - 1})',
                    fontsize=10, fontweight='bold')
            # Legend on top-right panel only
            if row == 0 and col == 2:
                ax.legend(handles=CATEGORY_PATCHES, loc='lower right',
                          fontsize=9, framealpha=0.9)
            _set_ax_style(ax)

    fig.suptitle(
        'Combined Feature Importance — All 17 Features × All 4 Phases\n'
        'LGBM Gain  |  Mean |SHAP|  |  Permutation GB-LGBM',
        fontsize=14, fontweight='bold')

    fname = os.path.join(PLOTS_DIR,
                         f'combined_feature_importance_{timestamp}.png')
    fig.savefig(fname, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Combined feature importance figure saved: {fname}")
    plt.show()
    plt.close(fig)

    # ── Serialisation helpers ─────────────────────────────────────────────────
    def _safe(v):
        if isinstance(v, (np.integer, np.int64)):    return int(v)
        if isinstance(v, (np.floating, np.float64)): return float(v)
        if isinstance(v, np.ndarray):                return v.tolist()
        return v

    def _clean(d):
        if isinstance(d, dict):  return {k: _clean(v) for k, v in d.items()}
        if isinstance(d, list):  return [_clean(i) for i in d]
        return _safe(d)

    # ── JSON ──────────────────────────────────────────────────────────────────
    fi_bundle = {
        'timestamp':          timestamp,
        'window_length':      WINDOW_LENGTH,
        'n_features':         len(ALL_FEATURES),
        'all_features':       ALL_FEATURES,
        'feature_categories': {k: v for k, v in FEATURE_CATEGORIES.items()},
        'phases':             _clean({str(k): v for k, v in fi_numeric.items()}),
    }
    fi_json = os.path.join(DATA_DIR, f'feature_importance_{timestamp}.json')
    with open(fi_json, 'w', encoding='utf-8') as fh:
        json.dump(fi_bundle, fh, indent=2, ensure_ascii=False)
    print(f"Feature importance JSON saved: {fi_json}")

    # ── TXT summary ───────────────────────────────────────────────────────────
    fi_txt = os.path.join(DATA_DIR, f'feature_importance_summary_{timestamp}.txt')
    sep80  = "="*80 + "\n"
    sep60  = "-"*60 + "\n"
    with open(fi_txt, 'w', encoding='utf-8') as fh:
        fh.write(sep80)
        fh.write("FEATURE IMPORTANCE SUMMARY - GB-LGBM HYBRID\n")
        fh.write(sep80)
        fh.write(f"Timestamp     : {timestamp}\n")
        fh.write(f"Window length : {WINDOW_LENGTH} frames\n")
        fh.write(f"N features    : {len(ALL_FEATURES)}\n\n")
        for phase in sorted(fi_numeric.keys()):
            fh.write(sep80)
            fh.write(f"PHASE {phase}: frames "
                     f"{BEST_CONFIGS[phase]['start_frame']}-"
                     f"{BEST_CONFIGS[phase]['start_frame'] + WINDOW_LENGTH - 1}\n")
            fh.write(sep80)
            fh.write(f"  {'Feature':>35}  {'Gain(norm)':>12}  "
                     f"{'MeanSHAP':>10}  {'Perm_GBLGBM':>12}\n")
            fh.write(sep60)
            pn   = fi_numeric[phase]
            rows = sorted(
                zip(pn['features'], pn['gain'], pn['shap_mean'],
                    pn['perm_gblgbm_mean']),
                key=lambda r: r[2], reverse=True)
            for feat, g, s, pg in rows:
                dname = FEATURE_DISPLAY_NAMES.get(feat, feat)
                cat   = _feature_category(feat)
                fh.write(f"  [{cat:>14}] {dname:>28}  "
                         f"{g:>12.6f}  {s:>10.6f}  {pg:>12.6f}\n")
            fh.write("\n")
    print(f"Feature importance TXT saved: {fi_txt}")
    return fname, fi_json, fi_txt


print("\n" + "="*80)
print("CELL 37: COMBINED FEATURE IMPORTANCE COMPARISON + SAVE RESULTS")
print("="*80)

_cfi_fig, _fi_json, _fi_txt = plot_combined_feature_importance(
    FI_DATA, SHAP_STORE, TIMESTAMP)

print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS COMPLETE")
print("="*80)